In [1]:
!wget -q https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_heavy/float16/1/pose_landmarker_heavy.task -O pose_landmarker.task

import os
print("Model downloaded:" , os.path.exists("pose_landmarker.task"))
print("File size:", round(os.path.getsize("pose_landmarker.task") / 1e6, 1), "MB")

Model downloaded: True
File size: 30.7 MB


In [2]:
!pip install -q mediapipe

import os
import cv2
import numpy as np
from collections import deque
import json
from datetime import datetime

from mediapipe.tasks import python
from mediapipe.tasks.python import vision
from mediapipe.tasks.python.vision.core.image import Image as MpImage
from mediapipe.tasks.python.vision.core.image import ImageFormat

# -------------------- PATHS --------------------
CALIBRATION_VIDEO = "/kaggle/input/datasets/sarangsharma/balance-axspa/Balance_cal.mp4"
TEST_VIDEO        = "/kaggle/input/datasets/sarangsharma/balance-axspa/balance_1.mp4"
OUTPUT_DIR        = "/kaggle/working/"
MODEL_PATH        = "pose_landmarker.task"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Scale (165 cm / ~720 px)
SCALE_CM_PER_PIXEL = 0.2292

# -------------------- Settings --------------------
LEFT_HIP, RIGHT_HIP = 23, 24
LEFT_ANKLE, RIGHT_ANKLE = 27, 28
LEFT_HEEL, RIGHT_HEEL = 29, 30
TRAIL_LENGTH = 120

def get_hip_center(landmarks, w, h):
    lh = landmarks[LEFT_HIP]
    rh = landmarks[RIGHT_HIP]
    return np.array([(lh.x + rh.x) * 0.5 * w,
                     (lh.y + rh.y) * 0.5 * h], dtype=np.float64)

def create_landmarker():
    base_options = python.BaseOptions(model_asset_path=MODEL_PATH)
    options = vision.PoseLandmarkerOptions(
        base_options=base_options,
        running_mode=vision.RunningMode.VIDEO,
        num_poses=1,
        min_pose_detection_confidence=0.6,
        min_pose_presence_confidence=0.6,
        min_tracking_confidence=0.6
    )
    return vision.PoseLandmarker.create_from_options(options)

def process_calibration(video_path):
    print(f"\n[Calibration] Processing: {os.path.basename(video_path)}")
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise FileNotFoundError(video_path)

    landmarker = create_landmarker()
    centres = []
    frame_idx = 0
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        h, w = frame.shape[:2]
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        mp_image = MpImage(image_format=ImageFormat.SRGB, data=rgb)
        result = landmarker.detect_for_video(mp_image, int(frame_idx * 1000 / fps))
        if result.pose_landmarks:
            centres.append(get_hip_center(result.pose_landmarks[0], w, h))
        frame_idx += 1

    cap.release()
    landmarker.close()

    if len(centres) < 10:
        raise RuntimeError("Too few valid detections in calibration video.")
    mean_c = np.mean(np.array(centres), axis=0)
    print(f"[Calibration] Mean centre (px): x={mean_c[0]:.1f}, y={mean_c[1]:.1f}")
    return mean_c

def analyse_with_segmentation(test_path, mean_center, output_video_path):
    print(f"\n[Test + Segmentation] Processing: {os.path.basename(test_path)}")
    cap = cv2.VideoCapture(test_path)
    if not cap.isOpened():
        raise FileNotFoundError(test_path)

    fps    = cap.get(cv2.CAP_PROP_FPS) or 30.0
    dt     = 1.0 / fps
    width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    out    = cv2.VideoWriter(output_video_path, fourcc, fps, (width, height))

    landmarker = create_landmarker()
    trail = deque(maxlen=TRAIL_LENGTH)

    # Storage for full trial + per-frame data
    all_x = []
    all_ml = []
    all_times = []
    all_stance = []          # 2 = double, 1 = single
    velocities = []
    prev_x = None
    ml_path_length = 0.0
    max_abs_ml = 0.0
    valid_frames = 0

    scale = SCALE_CM_PER_PIXEL
    frame_idx = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        t = frame_idx * dt
        h, w = frame.shape[:2]
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        mp_image = MpImage(image_format=ImageFormat.SRGB, data=rgb)
        result = landmarker.detect_for_video(mp_image, int(t * 1000))

        current_ml = 0.0
        stance_label = 2   # default double

        if result.pose_landmarks:
            lm = result.pose_landmarks[0]
            center = get_hip_center(lm, w, h)
            trail.append(center.copy())
            x = center[0]

            ml_px = x - mean_center[0]
            current_ml = ml_px * scale
            abs_ml = abs(current_ml)

            if abs_ml > max_abs_ml:
                max_abs_ml = abs_ml
            all_x.append(x)
            all_ml.append(current_ml)
            all_times.append(t)

            # Simple foot detection for stance classification
            feet_visible = 0
            for idx in [LEFT_ANKLE, RIGHT_ANKLE, LEFT_HEEL, RIGHT_HEEL]:
                if lm[idx].visibility > 0.55:
                    feet_visible += 0.5
            stance_label = 2 if feet_visible >= 1.5 else 1
            all_stance.append(stance_label)

            if prev_x is not None:
                dx = (x - prev_x) * scale
                ml_path_length += abs(dx)
                vel = abs(dx) / dt
                velocities.append(vel)
            prev_x = x
            valid_frames += 1

            # Draw trail + centre
            pts = list(trail)
            for i in range(1, len(pts)):
                alpha = i / len(pts)
                color = (0, int(180 * alpha), int(255 * alpha))
                cv2.line(frame, (int(pts[i-1][0]), int(pts[i-1][1])),
                         (int(pts[i][0]), int(pts[i][1])), color, 2)
            cv2.circle(frame, (int(center[0]), int(center[1])), 8, (0, 255, 255), -1)

        # Live overlay
        overlay = frame.copy()
        cv2.rectangle(overlay, (8, 8), (470, 210), (0, 0, 0), -1)
        cv2.addWeighted(overlay, 0.55, frame, 0.45, 0, frame)

        ml_range_now = (max(all_x) - min(all_x)) * scale if all_x else 0.0
        duration_so_far = valid_frames / fps if fps > 0 else 0.0
        mean_vel_now = ml_path_length / duration_so_far if duration_so_far > 0 else 0.0
        peak_vel_now = max(velocities) if velocities else 0.0

        font = cv2.FONT_HERSHEY_SIMPLEX
        y0, dy = 35, 26
        cv2.putText(frame, f"ML disp:   {current_ml:+6.1f} cm", (20, y0), font, 0.65, (0, 255, 255), 2)
        cv2.putText(frame, f"Max ML:    {max_abs_ml:6.1f} cm", (20, y0+dy), font, 0.65, (0, 255, 120), 2)
        cv2.putText(frame, f"ML range:  {ml_range_now:6.1f} cm", (20, y0+2*dy), font, 0.65, (255, 200, 50), 2)
        cv2.putText(frame, f"ML path:   {ml_path_length:6.1f} cm", (20, y0+3*dy), font, 0.65, (255, 160, 50), 2)
        cv2.putText(frame, f"Mean vel:  {mean_vel_now:6.1f} cm/s", (20, y0+4*dy), font, 0.65, (180, 180, 255), 2)
        cv2.putText(frame, f"Peak vel:  {peak_vel_now:6.1f} cm/s", (20, y0+5*dy), font, 0.65, (200, 140, 255), 2)

        # Show current stance label
        stance_txt = "Double" if stance_label == 2 else "Single"
        cv2.putText(frame, f"Stance: {stance_txt}", (20, y0+6*dy+5), font, 0.65, (0, 220, 255), 2)

        out.write(frame)
        frame_idx += 1

    cap.release()
    out.release()
    landmarker.close()

    # -------------------- Segmentation --------------------
   # -------------------- REPLACE the segmentation part with this --------------------

def analyse_with_segmentation(test_path, mean_center, output_video_path):
    print(f"\n[Test + Segmentation] Processing: {os.path.basename(test_path)}")
    cap = cv2.VideoCapture(test_path)
    if not cap.isOpened():
        raise FileNotFoundError(test_path)

    fps    = cap.get(cv2.CAP_PROP_FPS) or 30.0
    dt     = 1.0 / fps
    width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    out    = cv2.VideoWriter(output_video_path, fourcc, fps, (width, height))

    landmarker = create_landmarker()
    trail = deque(maxlen=TRAIL_LENGTH)

    all_x, all_ml, all_times = [], [], []
    velocities = []
    prev_x = None
    ml_path_length = 0.0
    max_abs_ml = 0.0
    valid_frames = 0
    scale = SCALE_CM_PER_PIXEL
    frame_idx = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        t = frame_idx * dt
        h, w = frame.shape[:2]
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        mp_image = MpImage(image_format=ImageFormat.SRGB, data=rgb)
        result = landmarker.detect_for_video(mp_image, int(t * 1000))

        current_ml = 0.0

        if result.pose_landmarks:
            lm = result.pose_landmarks[0]
            center = get_hip_center(lm, w, h)
            trail.append(center.copy())
            x = center[0]

            ml_px = x - mean_center[0]
            current_ml = ml_px * scale
            abs_ml = abs(current_ml)

            if abs_ml > max_abs_ml:
                max_abs_ml = abs_ml

            all_x.append(x)
            all_ml.append(current_ml)
            all_times.append(t)

            if prev_x is not None:
                dx = (x - prev_x) * scale
                ml_path_length += abs(dx)
                velocities.append(abs(dx) / dt)
            prev_x = x
            valid_frames += 1

            # Draw trail + centre
            pts = list(trail)
            for i in range(1, len(pts)):
                alpha = i / len(pts)
                color = (0, int(180 * alpha), int(255 * alpha))
                cv2.line(frame, (int(pts[i-1][0]), int(pts[i-1][1])),
                         (int(pts[i][0]), int(pts[i][1])), color, 2)
            cv2.circle(frame, (int(center[0]), int(center[1])), 8, (0, 255, 255), -1)

        # Live overlay
        overlay = frame.copy()
        cv2.rectangle(overlay, (8, 8), (470, 195), (0, 0, 0), -1)
        cv2.addWeighted(overlay, 0.55, frame, 0.45, 0, frame)

        ml_range_now = (max(all_x) - min(all_x)) * scale if all_x else 0.0
        duration_so_far = valid_frames / fps if fps > 0 else 0.0
        mean_vel_now = ml_path_length / duration_so_far if duration_so_far > 0 else 0.0
        peak_vel_now = max(velocities) if velocities else 0.0

        font = cv2.FONT_HERSHEY_SIMPLEX
        y0, dy = 35, 26
        cv2.putText(frame, f"ML disp:   {current_ml:+6.1f} cm", (20, y0), font, 0.65, (0, 255, 255), 2)
        cv2.putText(frame, f"Max ML:    {max_abs_ml:6.1f} cm", (20, y0+dy), font, 0.65, (0, 255, 120), 2)
        cv2.putText(frame, f"ML range:  {ml_range_now:6.1f} cm", (20, y0+2*dy), font, 0.65, (255, 200, 50), 2)
        cv2.putText(frame, f"ML path:   {ml_path_length:6.1f} cm", (20, y0+3*dy), font, 0.65, (255, 160, 50), 2)
        cv2.putText(frame, f"Mean vel:  {mean_vel_now:6.1f} cm/s", (20, y0+4*dy), font, 0.65, (180, 180, 255), 2)
        cv2.putText(frame, f"Peak vel:  {peak_vel_now:6.1f} cm/s", (20, y0+5*dy), font, 0.65, (200, 140, 255), 2)

        # Time-based stance label (most reliable for your protocol)
        if t < 10.0:
            stance_txt = "Initial_Position"
        elif t < 20.0:
            stance_txt = "Semi_tandem"
        else:
            stance_txt = "Full_tandem"
        cv2.putText(frame, f"Stance: {stance_txt}", (20, y0+6*dy+5), font, 0.65, (0, 220, 255), 2)

        out.write(frame)
        frame_idx += 1

    cap.release()
    out.release()
    landmarker.close()

    # -------------------- Reliable time-based segmentation --------------------
    # Because recording was paused between stances, we use fixed 10-second windows
    total_duration = all_times[-1] if all_times else 0.0

    stance_windows = [
        {"name": "Initial_Position",   "start": 0.0,  "end": 10.0},
        {"name": "Semi_tandem", "start": 10.0, "end": 20.0},
        {"name": "Full_tandem", "start": 20.0, "end": total_duration + 0.1}
    ]

    segments = []
    for win in stance_windows:
        # Find frames belonging to this window
        idxs = [i for i, t in enumerate(all_times) if win["start"] <= t < win["end"]]
        if len(idxs) < 10:
            continue

        seg_ml = [all_ml[i] for i in idxs]
        seg_x  = [all_x[i] for i in idxs]
        start_t = all_times[idxs[0]]
        end_t   = all_times[idxs[-1]]
        duration = end_t - start_t

        max_ml = max(abs(m) for m in seg_ml)
        range_ml = (max(seg_x) - min(seg_x)) * scale
        path = sum(abs(seg_x[j] - seg_x[j-1]) for j in range(1, len(seg_x))) * scale
        mean_v = path / duration if duration > 0 else 0.0

        segments.append({
            "stance": win["name"],
            "start_s": round(start_t, 2),
            "end_s": round(end_t, 2),
            "duration_s": round(duration, 2),
            "max_ml_cm": round(max_ml, 2),
            "ml_range_cm": round(range_ml, 2),
            "ml_path_cm": round(path, 2),
            "mean_vel_cm_s": round(mean_v, 2)
        })

    # Overall metrics
    duration = valid_frames / fps if valid_frames > 1 else 0.0
    ml_range = (max(all_x) - min(all_x)) * scale if all_x else 0.0
    mean_vel = ml_path_length / duration if duration > 0 else 0.0
    peak_vel = max(velocities) if velocities else 0.0
    filtered_vel = [v for v in velocities if v < 40.0]
    peak_vel_f = max(filtered_vel) if filtered_vel else 0.0

    metrics = {
        "overall": {
            "max_ml_displacement_cm": round(max_abs_ml, 2),
            "ml_sway_range_cm": round(ml_range, 2),
            "ml_path_length_cm": round(ml_path_length, 2),
            "mean_ml_velocity_cm_s": round(mean_vel, 2),
            "peak_ml_velocity_cm_s": round(peak_vel, 2),
            "peak_ml_velocity_filtered_cm_s": round(peak_vel_f, 2),
            "duration_s": round(duration, 2),
            "valid_frames": valid_frames
        },
        "stances": segments
    }

    # Print
    print("\n" + "="*70)
    print("OVERALL ML METRICS")
    print("="*70)
    for k, v in metrics["overall"].items():
        print(f"{k:40s}: {v}")

    print("\n" + "="*70)
    print("PER-STANCE METRICS (10-second windows)")
    print("="*70)
    for i, seg in enumerate(segments, 1):
        print(f"\nStance {i}: {seg['stance']}")
        print(f"  Time        : {seg['start_s']:.2f}s → {seg['end_s']:.2f}s  ({seg['duration_s']:.2f}s)")
        print(f"  Max ML      : {seg['max_ml_cm']:.2f} cm")
        print(f"  ML Range    : {seg['ml_range_cm']:.2f} cm")
        print(f"  Path length : {seg['ml_path_cm']:.2f} cm")
        print(f"  Mean vel    : {seg['mean_vel_cm_s']:.2f} cm/s")

    metrics_path = os.path.join(OUTPUT_DIR, "ml_balance_metrics_segmented.json")
    with open(metrics_path, "w") as f:
        json.dump(metrics, f, indent=2)

    print(f"\nMetrics saved  → {metrics_path}")
    print(f"Annotated video → {output_video_path}")
    return metrics

# -------------------- RUN --------------------
print("Starting full analysis with stance segmentation...")
print(f"Scale = {SCALE_CM_PER_PIXEL:.4f} cm/pixel")

mean_c = process_calibration(CALIBRATION_VIDEO)
output_video = os.path.join(OUTPUT_DIR, "annotated_balance_1_segmented.mp4")
metrics = analyse_with_segmentation(TEST_VIDEO, mean_c, output_video)

print("\nDone!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.9/37.9 MB 51.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.4/137.4 kB 9.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
Starting full analysis with stance segmentation...
Scale = 0.2292 cm/pixel

[Calibration] Processing: Balance_cal.mp4


INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1787501824.019739      89 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1787501824.114551      90 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1787501824.233992      91 landmark_projection_calculator.cc:81] Using NORM_RECT without IMAGE_DIMENSIONS is only supported for the square ROI. Provide IMAGE_DIMENSIONS or use PROJECTION_MATRIX.


[Calibration] Mean centre (px): x=280.0, y=580.1

[Test + Segmentation] Processing: balance_1.mp4


W0000 00:00:1787501846.191023     104 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1787501846.247908     106 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.



OVERALL ML METRICS
max_ml_displacement_cm                  : 4.29
ml_sway_range_cm                        : 8.57
ml_path_length_cm                       : 17.37
mean_ml_velocity_cm_s                   : 0.57
peak_ml_velocity_cm_s                   : 153.71
peak_ml_velocity_filtered_cm_s          : 15.97
duration_s                              : 30.57
valid_frames                            : 917

PER-STANCE METRICS (10-second windows)

Stance 1: Initial_Position
  Time        : 0.00s → 9.97s  (9.97s)
  Max ML      : 1.66 cm
  ML Range    : 0.67 cm
  Path length : 2.05 cm
  Mean vel    : 0.21 cm/s

Stance 2: Semi_tandem
  Time        : 10.00s → 19.97s  (9.97s)
  Max ML      : 4.28 cm
  ML Range    : 2.93 cm
  Path length : 4.47 cm
  Mean vel    : 0.45 cm/s

Stance 3: Full_tandem
  Time        : 20.00s → 30.53s  (10.53s)
  Max ML      : 4.29 cm
  ML Range    : 7.90 cm
  Path length : 10.83 cm
  Mean vel    : 1.03 cm/s

Metrics saved  → /kaggle/working/ml_balance_metrics_segmented.json
A

In [3]:
# ============================================================
# Stand-and-Reach Test – Final Version with Clean Dashboard
# Right × 2 → Left × 2
# Clean video overlay + Professional results table
# ============================================================

!pip install -q mediapipe opencv-python scipy pandas

import cv2
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
import numpy as np
import pandas as pd
from scipy.signal import savgol_filter, find_peaks
from pathlib import Path
import urllib.request
import os

# ----------------------------- USER CONFIG -----------------------------
VIDEO_PATH = "/kaggle/input/datasets/sarangsharma/stand-and-reach/Reach.mp4"
SUBJECT_HEIGHT_CM = 165.0

OUTPUT_DIR = Path("/kaggle/working")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_VIDEO = OUTPUT_DIR / "stand_reach_FINAL_annotated.mp4"
OUTPUT_CSV   = OUTPUT_DIR / "stand_reach_FINAL_metrics.csv"
# -----------------------------------------------------------------------

MODEL_URL  = "https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_lite/float16/1/pose_landmarker_lite.task"
MODEL_PATH = "/kaggle/working/pose_landmarker_lite.task"
if not os.path.exists(MODEL_PATH):
    print("Downloading Pose Landmarker model...")
    urllib.request.urlretrieve(MODEL_URL, MODEL_PATH)

BaseOptions = python.BaseOptions
PoseLandmarker = vision.PoseLandmarker
PoseLandmarkerOptions = vision.PoseLandmarkerOptions
VisionRunningMode = vision.RunningMode

options = PoseLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=MODEL_PATH),
    running_mode=VisionRunningMode.VIDEO,
    num_poses=1,
    min_pose_detection_confidence=0.5,
    min_pose_presence_confidence=0.5,
    min_tracking_confidence=0.5
)

def elevation_angle(shoulder, elbow, mid_hip, mid_sh):
    torso = mid_hip - mid_sh
    arm   = elbow - shoulder
    torso = torso / (np.linalg.norm(torso) + 1e-8)
    arm   = arm   / (np.linalg.norm(arm)   + 1e-8)
    cosang = np.clip(np.dot(torso, arm), -1.0, 1.0)
    return np.degrees(np.arccos(cosang))

def get_px(lm, w, h):
    return np.array([lm.x * w, lm.y * h])

def process_video(video_path, height_cm):
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise FileNotFoundError(f"Cannot open video: {video_path}")

    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    w   = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h   = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    print(f"Video: {w}×{h} @ {fps:.1f} fps\n")

    times, L_flex, R_flex, L_reach, R_reach = [], [], [], [], []

    with PoseLandmarker.create_from_options(options) as landmarker:
        idx = 0
        while True:
            ret, frame = cap.read()
            if not ret:
                break

            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            mp_img = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
            ts = int(idx * 1000 / fps)
            res = landmarker.detect_for_video(mp_img, ts)

            t = idx / fps
            lf = rf = lr = rr = np.nan

            if res.pose_landmarks:
                lms = res.pose_landmarks[0]

                nose  = get_px(lms[0],  w, h)
                l_sh  = get_px(lms[11], w, h)
                r_sh  = get_px(lms[12], w, h)
                l_el  = get_px(lms[13], w, h)
                r_el  = get_px(lms[14], w, h)
                l_wr  = get_px(lms[15], w, h)
                r_wr  = get_px(lms[16], w, h)
                l_hip = get_px(lms[23], w, h)
                r_hip = get_px(lms[24], w, h)
                l_ank = get_px(lms[27], w, h)
                r_ank = get_px(lms[28], w, h)

                mid_sh  = (l_sh + r_sh) / 2
                mid_hip = (l_hip + r_hip) / 2
                mid_ank = (l_ank + r_ank) / 2

                head_top_y = nose[1] - 0.08 * abs(mid_ank[1] - nose[1])
                body_px = abs(mid_ank[1] - head_top_y)
                scale = height_cm / body_px if body_px > 50 else np.nan

                lf = elevation_angle(l_sh, l_el, mid_hip, mid_sh)
                rf = elevation_angle(r_sh, r_el, mid_hip, mid_sh)

                if not np.isnan(scale):
                    lr = (mid_ank[1] - l_wr[1]) * scale
                    rr = (mid_ank[1] - r_wr[1]) * scale

            times.append(t)
            L_flex.append(lf)
            R_flex.append(rf)
            L_reach.append(lr)
            R_reach.append(rr)
            idx += 1

    cap.release()

    times   = np.asarray(times)
    L_flex  = np.asarray(L_flex,  dtype=float)
    R_flex  = np.asarray(R_flex,  dtype=float)
    L_reach = np.asarray(L_reach, dtype=float)
    R_reach = np.asarray(R_reach, dtype=float)

    def fill(a):
        n = np.isnan(a)
        if n.any() and (~n).any():
            a[n] = np.interp(np.flatnonzero(n), np.flatnonzero(~n), a[~n])
        return a

    L_flex  = fill(L_flex)
    R_flex  = fill(R_flex)
    L_reach = fill(L_reach)
    R_reach = fill(R_reach)

    win = max(5, int(0.30 * fps) | 1)
    L_flex_s  = savgol_filter(L_flex,  win, 2)
    R_flex_s  = savgol_filter(R_flex,  win, 2)
    L_reach_s = savgol_filter(L_reach, win, 2)
    R_reach_s = savgol_filter(R_reach, win, 2)

    def make_relative(reach):
        baseline = np.nanpercentile(reach, 12)
        return np.maximum(reach - baseline, 0)

    L_reach_rel = make_relative(L_reach_s)
    R_reach_rel = make_relative(R_reach_s)

    # -------------------- Peak Detection --------------------
    min_dist = int(0.9 * fps)

    def find_top2_raises(flex_signal, reach_signal):
        peaks, props = find_peaks(
            flex_signal,
            height=55,
            distance=min_dist,
            prominence=12
        )
        if len(peaks) == 0:
            return []
        scores = flex_signal[peaks] + 0.25 * reach_signal[peaks]
        order = np.argsort(scores)[::-1][:2]
        selected = sorted(peaks[order])
        return selected

    right_peaks = find_top2_raises(R_flex_s, R_reach_rel)
    left_peaks  = find_top2_raises(L_flex_s, L_reach_rel)

    all_events = []
    for i, p in enumerate(right_peaks):
        all_events.append(("right", p, f"Right_T{i+1}"))
    for i, p in enumerate(left_peaks):
        all_events.append(("left",  p, f"Left_T{i+1}"))
    all_events.sort(key=lambda x: x[1])

    half = int(1.0 * fps)
    summary = []

    for side, peak_idx, label in all_events:
        s = max(0, peak_idx - half)
        e = min(len(times) - 1, peak_idx + half)

        if side == "right":
            flex_seg  = R_flex_s[s:e+1]
            reach_seg = R_reach_rel[s:e+1]
        else:
            flex_seg  = L_flex_s[s:e+1]
            reach_seg = L_reach_rel[s:e+1]

        peak_flex  = round(float(np.nanmax(flex_seg)), 1)
        peak_reach = round(float(np.nanmax(reach_seg)), 1)

        # Simple quality status
        status = "Good" if peak_flex >= 150 else "Fair"

        summary.append({
            "Trial": label,
            "Time": f"{times[peak_idx]:.2f} s",
            "Peak Flexion": f"{peak_flex}°",
            "Peak Reach (relative)": f"{peak_reach} cm",
            "Status": status
        })

    summary_df = pd.DataFrame(summary)

    # -------------------- Professional Dashboard --------------------
    print("=" * 72)
    print("               STAND-AND-REACH TEST – RESULTS DASHBOARD")
    print("=" * 72)
    print(f"{'Trial':<12} {'Time':<10} {'Peak Flexion':<16} {'Peak Reach (relative)':<22} {'Status'}")
    print("-" * 72)
    for r in summary:
        print(f"{r['Trial']:<12} {r['Time']:<10} {r['Peak Flexion']:<16} {r['Peak Reach (relative)']:<22} {r['Status']}")
    print("=" * 72)
    print()

    # -------------------- CSV --------------------
    df = pd.DataFrame({
        "frame": np.arange(len(times)),
        "time_s": np.round(times, 3),
        "left_flexion_deg":  np.round(L_flex_s, 1),
        "right_flexion_deg": np.round(R_flex_s, 1),
        "left_reach_cm":  np.round(L_reach_rel, 1),
        "right_reach_cm": np.round(R_reach_rel, 1)
    })

    with open(OUTPUT_CSV, "w") as f:
        f.write("# Per-frame data\n")
        f.write("# Units: flexion = deg, reach = cm (relative to resting position)\n")
    df.to_csv(OUTPUT_CSV, mode="a", index=False)

    with open(OUTPUT_CSV, "a") as f:
        f.write("\n# Summary of detected trials\n")
    summary_df.to_csv(OUTPUT_CSV, mode="a", index=False)

    print(f"CSV saved → {OUTPUT_CSV}")

    # -------------------- Clean Annotated Video --------------------
    cap = cv2.VideoCapture(str(video_path))
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    out = cv2.VideoWriter(str(OUTPUT_VIDEO), fourcc, fps, (w, h))

    with PoseLandmarker.create_from_options(options) as landmarker:
        idx = 0
        while True:
            ret, frame = cap.read()
            if not ret:
                break

            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            mp_img = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
            res = landmarker.detect_for_video(mp_img, int(idx * 1000 / fps))

            if res.pose_landmarks:
                for i in [0, 11, 12, 13, 14, 15, 16, 23, 24, 27, 28]:
                    x = int(res.pose_landmarks[0][i].x * w)
                    y = int(res.pose_landmarks[0][i].y * h)
                    cv2.circle(frame, (x, y), 5, (0, 255, 0), -1)

            overlay = frame.copy()
            cv2.rectangle(overlay, (8, 8), (510, 95), (15, 15, 15), -1)
            cv2.addWeighted(overlay, 0.65, frame, 0.35, 0, frame)

            left_val  = L_flex_s[idx]
            right_val = R_flex_s[idx]

            if right_val >= left_val and right_val > 45:
                text = f"RIGHT  Flex: {R_flex_s[idx]:5.1f} deg    Reach: {R_reach_rel[idx]:5.1f} cm"
                color = (0, 255, 120)
            elif left_val > right_val and left_val > 45:
                text = f"LEFT   Flex: {L_flex_s[idx]:5.1f} deg    Reach: {L_reach_rel[idx]:5.1f} cm"
                color = (0, 255, 255)
            else:
                text = "Arms down   Flex ≈ 0 deg    Reach ≈ 0 cm"
                color = (170, 170, 170)

            cv2.putText(frame, text, (18, 55),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.72, color, 2)

            out.write(frame)
            idx += 1

    cap.release()
    out.release()
    print(f"Annotated video saved → {OUTPUT_VIDEO}\n")

    return summary_df

# ===================== RUN =====================
summary = process_video(VIDEO_PATH, SUBJECT_HEIGHT_CM)

Video: 576×1024 @ 30.0 fps



W0000 00:00:1787501916.506509     123 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1787501916.530876     124 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


               STAND-AND-REACH TEST – RESULTS DASHBOARD
Trial        Time       Peak Flexion     Peak Reach (relative)  Status
------------------------------------------------------------------------
Right_T1     1.00 s     161.9°           130.8 cm               Good
Right_T2     2.13 s     165.7°           133.1 cm               Good
Left_T1      3.60 s     166.8°           129.8 cm               Good
Left_T2      5.00 s     173.7°           131.2 cm               Good

CSV saved → /kaggle/working/stand_reach_FINAL_metrics.csv


W0000 00:00:1787501920.450389     136 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1787501920.480467     135 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


Annotated video saved → /kaggle/working/stand_reach_FINAL_annotated.mp4



In [4]:
# ============================================================
# Put-on-Socks Test – 4 DISTINCT TRIALS
# Fixed: degree unit display + correct hip/knee flexion direction
# ============================================================

!pip install -q mediapipe opencv-python-headless pandas matplotlib scipy

import cv2
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import os
from pathlib import Path
import urllib.request

# ------------------------------
# 1. CONFIGURATION
# ------------------------------
VIDEO_PATH = "/kaggle/input/datasets/sarangsharma/socks-put/putonsocks.mp4"
OUTPUT_DIR = "/kaggle/working/socks_results"
OUTPUT_VIDEO = os.path.join(OUTPUT_DIR, "put_on_socks_annotated.mp4")
OUTPUT_CSV   = os.path.join(OUTPUT_DIR, "put_on_socks_metrics.csv")
DASHBOARD_PNG = os.path.join(OUTPUT_DIR, "results_dashboard.png")
MODEL_PATH = "/kaggle/working/pose_landmarker_full.task"

EMA_ALPHA = 0.25
MIN_VISIBILITY = 0.5

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ------------------------------
# 2. TRIAL WINDOWS (your observed start frames)
# ------------------------------
TRIAL_STARTS = {
    1: 324,   # Right Rep 1
    2: 502,   # Right Rep 2
    3: 649,   # Left  Rep 1
    4: 779,   # Left  Rep 2
}
TRIAL_LABELS = {
    1: "Right - Rep 1",
    2: "Right - Rep 2",
    3: "Left  - Rep 1",
    4: "Left  - Rep 2",
}

# ------------------------------
# 3. DOWNLOAD MODEL
# ------------------------------
if not os.path.exists(MODEL_PATH):
    print("Downloading Pose Landmarker model...")
    url = "https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_full/float16/1/pose_landmarker_full.task"
    urllib.request.urlretrieve(url, MODEL_PATH)
    print("Model downloaded.")

# ------------------------------
# 4. DRAWING (pure OpenCV)
# ------------------------------
POSE_CONNECTIONS = [
    (0, 1), (1, 2), (2, 3), (3, 7), (0, 4), (4, 5), (5, 6), (6, 8),
    (9, 10),
    (11, 12), (11, 23), (12, 24), (23, 24),
    (11, 13), (13, 15), (15, 17), (15, 19), (15, 21), (17, 19),
    (12, 14), (14, 16), (16, 18), (16, 20), (16, 22), (18, 20),
    (23, 25), (25, 27), (27, 29), (27, 31), (29, 31),
    (24, 26), (26, 28), (28, 30), (28, 32), (30, 32),
]

def draw_pose(image_bgr, landmarks, w, h,
              connection_color=(0, 255, 0), landmark_color=(0, 0, 255),
              thickness=2, radius=3):
    for s, e in POSE_CONNECTIONS:
        if s < len(landmarks) and e < len(landmarks):
            lm1, lm2 = landmarks[s], landmarks[e]
            if lm1.visibility > 0.5 and lm2.visibility > 0.5:
                pt1 = (int(lm1.x * w), int(lm1.y * h))
                pt2 = (int(lm2.x * w), int(lm2.y * h))
                cv2.line(image_bgr, pt1, pt2, connection_color, thickness, cv2.LINE_AA)
    for lm in landmarks:
        if lm.visibility > 0.5:
            cv2.circle(image_bgr, (int(lm.x * w), int(lm.y * h)),
                       radius, landmark_color, -1, cv2.LINE_AA)

# ------------------------------
# 5. ANGLE HELPERS
# ------------------------------
def joint_angle(a, b, c):
    """Interior angle at b (0-180)."""
    a = np.array(a, dtype=np.float64)
    b = np.array(b, dtype=np.float64)
    c = np.array(c, dtype=np.float64)
    ba = a - b
    bc = c - b
    cosine = np.dot(ba, bc) / (np.linalg.norm(ba) * np.linalg.norm(bc) + 1e-8)
    return np.degrees(np.arccos(np.clip(cosine, -1.0, 1.0)))

def hip_flexion(shoulder, hip, knee):
    """Flexion from full extension. Higher = more flexed.
    Seated ~40 deg, deep forward bend ~150-170 deg."""
    return 180.0 - joint_angle(shoulder, hip, knee)

def knee_flexion(hip, knee, ankle):
    """Flexion from full extension. Higher = more flexed."""
    return 180.0 - joint_angle(hip, knee, ankle)

def trunk_flexion_angle(mid_shoulder, mid_hip):
    """From vertical. 0 = upright, higher = more flexed."""
    vec = np.array(mid_shoulder) - np.array(mid_hip)
    vertical = np.array([0.0, -1.0])
    cos_theta = np.dot(vec, vertical) / (np.linalg.norm(vec) * np.linalg.norm(vertical) + 1e-8)
    return np.degrees(np.arccos(np.clip(cos_theta, -1.0, 1.0)))

def get_xy(landmark, w, h):
    return [landmark.x * w, landmark.y * h], landmark.visibility

class EMASmoother:
    def __init__(self, alpha=0.25):
        self.alpha = alpha
        self.value = None
    def update(self, new_val):
        if new_val is None:
            return self.value
        if self.value is None:
            self.value = new_val
        else:
            self.value = self.alpha * new_val + (1.0 - self.alpha) * self.value
        return self.value

def get_current_trial(frame_idx, starts):
    trial = 0
    for t in sorted(starts.keys()):
        if frame_idx >= starts[t]:
            trial = t
    return trial

# ------------------------------
# 6. LANDMARKER
# ------------------------------
BaseOptions = python.BaseOptions
PoseLandmarker = vision.PoseLandmarker
PoseLandmarkerOptions = vision.PoseLandmarkerOptions
VisionRunningMode = vision.RunningMode

options = PoseLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=MODEL_PATH),
    running_mode=VisionRunningMode.VIDEO,
    num_poses=1,
    min_pose_detection_confidence=0.5,
    min_pose_presence_confidence=0.5,
    min_tracking_confidence=0.5,
)

LS, RS, LH, RH, LK, RK, LA, RA = 11, 12, 23, 24, 25, 26, 27, 28

# ------------------------------
# 7. MAIN PROCESSING
# ------------------------------
cap = cv2.VideoCapture(VIDEO_PATH)
if not cap.isOpened():
    raise FileNotFoundError(f"Cannot open video: {VIDEO_PATH}")

fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

sorted_starts = sorted(TRIAL_STARTS.items())
TRIAL_ENDS = {}
for i, (t, start) in enumerate(sorted_starts):
    if i + 1 < len(sorted_starts):
        TRIAL_ENDS[t] = sorted_starts[i + 1][1] - 1
    else:
        TRIAL_ENDS[t] = total_frames - 1

print("Trial windows (frames):")
for t in range(1, 5):
    print(f"  Trial {t} ({TRIAL_LABELS[t]}): {TRIAL_STARTS[t]} -> {TRIAL_ENDS[t]}  "
          f"({TRIAL_STARTS[t]/fps:.2f}s - {TRIAL_ENDS[t]/fps:.2f}s)")

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
out = cv2.VideoWriter(OUTPUT_VIDEO, fourcc, fps, (width, height))

records = []
trial_peaks = {t: {"hip": 0.0, "knee": 0.0, "trunk": 0.0} for t in range(1, 5)}

smooth_hip_l  = EMASmoother(EMA_ALPHA)
smooth_hip_r  = EMASmoother(EMA_ALPHA)
smooth_knee_l = EMASmoother(EMA_ALPHA)
smooth_knee_r = EMASmoother(EMA_ALPHA)
smooth_trunk  = EMASmoother(EMA_ALPHA)

with PoseLandmarker.create_from_options(options) as landmarker:
    frame_idx = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
        timestamp_ms = int(frame_idx * 1000 / fps)
        result = landmarker.detect_for_video(mp_image, timestamp_ms)

        annotated = frame.copy()
        hip_l = hip_r = knee_l = knee_r = trunk = None
        vis_ok = False
        current_trial = get_current_trial(frame_idx, TRIAL_STARTS)

        if result.pose_landmarks:
            lms = result.pose_landmarks[0]
            draw_pose(annotated, lms, width, height)

            ls, v_ls = get_xy(lms[LS], width, height)
            rs, v_rs = get_xy(lms[RS], width, height)
            lh, v_lh = get_xy(lms[LH], width, height)
            rh, v_rh = get_xy(lms[RH], width, height)
            lk, v_lk = get_xy(lms[LK], width, height)
            rk, v_rk = get_xy(lms[RK], width, height)
            la, v_la = get_xy(lms[LA], width, height)
            ra, v_ra = get_xy(lms[RA], width, height)

            if min(v_ls, v_rs, v_lh, v_rh, v_lk, v_rk) > MIN_VISIBILITY:
                vis_ok = True
                mid_shoulder = [(ls[0] + rs[0]) / 2, (ls[1] + rs[1]) / 2]
                mid_hip      = [(lh[0] + rh[0]) / 2, (lh[1] + rh[1]) / 2]

                # Correct flexion definitions (higher = more flexed)
                hip_l  = hip_flexion(ls, lh, lk)
                hip_r  = hip_flexion(rs, rh, rk)
                knee_l = knee_flexion(lh, lk, la)
                knee_r = knee_flexion(rh, rk, ra)
                trunk  = trunk_flexion_angle(mid_shoulder, mid_hip)

                hip_l  = smooth_hip_l.update(hip_l)
                hip_r  = smooth_hip_r.update(hip_r)
                knee_l = smooth_knee_l.update(knee_l)
                knee_r = smooth_knee_r.update(knee_r)
                trunk  = smooth_trunk.update(trunk)

                if current_trial >= 1:
                    cur_hip  = max(hip_l or 0, hip_r or 0)
                    cur_knee = max(knee_l or 0, knee_r or 0)
                    trial_peaks[current_trial]["hip"]   = max(trial_peaks[current_trial]["hip"],   cur_hip)
                    trial_peaks[current_trial]["knee"]  = max(trial_peaks[current_trial]["knee"],  cur_knee)
                    trial_peaks[current_trial]["trunk"] = max(trial_peaks[current_trial]["trunk"], trunk or 0)

        # ---------- Overlay (ASCII-safe units) ----------
        overlay = annotated.copy()
        cv2.rectangle(overlay, (10, 10), (500, 245), (20, 20, 20), -1)
        cv2.addWeighted(overlay, 0.65, annotated, 0.35, 0, annotated)

        def put(txt, y, color=(240, 240, 240)):
            cv2.putText(annotated, txt, (25, y),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.58, color, 2, cv2.LINE_AA)

        put(f"Frame: {frame_idx:4d}  |  Time: {frame_idx/fps:5.2f}s", 38)

        trial_txt = TRIAL_LABELS.get(current_trial, "Baseline / Seated")
        put(f"Current: {trial_txt}", 68, (0, 220, 255))

        if hip_l is not None:
            put(f"Hip Flex   L: {hip_l:5.1f} deg   R: {hip_r:5.1f} deg", 100)
            put(f"Knee Flex  L: {knee_l:5.1f} deg   R: {knee_r:5.1f} deg", 130)
            put(f"Trunk Flex     : {trunk:5.1f} deg", 160)
        else:
            put("Hip Flex   --", 100)
            put("Knee Flex  --", 130)
            put("Trunk Flex --", 160)

        if current_trial >= 1:
            p = trial_peaks[current_trial]
            put(f"Trial peaks -> Hip {p['hip']:5.1f}  Knee {p['knee']:5.1f}  Trunk {p['trunk']:5.1f} deg",
                195, (0, 220, 120))
        else:
            put("Trial peaks -> (waiting for first movement)", 195, (180, 180, 180))

        cv2.putText(annotated, "Protocol: 2x Right -> 2x Left (seated put-on-socks)",
                    (15, height - 20), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (180, 180, 180), 1, cv2.LINE_AA)

        out.write(annotated)

        records.append({
            "frame": frame_idx,
            "time_s": round(frame_idx / fps, 3),
            "trial": current_trial,
            "trial_label": trial_txt,
            "hip_flexion_L": round(hip_l, 2) if hip_l is not None else np.nan,
            "hip_flexion_R": round(hip_r, 2) if hip_r is not None else np.nan,
            "knee_flexion_L": round(knee_l, 2) if knee_l is not None else np.nan,
            "knee_flexion_R": round(knee_r, 2) if knee_r is not None else np.nan,
            "trunk_flexion": round(trunk, 2) if trunk is not None else np.nan,
            "pose_detected": vis_ok
        })

        frame_idx += 1
        if frame_idx % 30 == 0:
            print(f"Processed {frame_idx}/{total_frames} frames...")

cap.release()
out.release()
print(f"\nAnnotated video saved -> {OUTPUT_VIDEO}")

# ------------------------------
# 8. CSV + PER-TRIAL SUMMARY
# ------------------------------
df = pd.DataFrame(records)
df.to_csv(OUTPUT_CSV, index=False)
print(f"CSV saved -> {OUTPUT_CSV}")

summary_rows = []
for t in range(1, 5):
    mask = df["trial"] == t
    if mask.any():
        sub = df.loc[mask]
        hip_peak   = float(np.nanmax(sub[["hip_flexion_L", "hip_flexion_R"]].values))
        knee_peak  = float(np.nanmax(sub[["knee_flexion_L", "knee_flexion_R"]].values))
        trunk_peak = float(np.nanmax(sub["trunk_flexion"].values))
        hip_peak   = max(hip_peak,   trial_peaks[t]["hip"])   if not np.isnan(hip_peak)   else trial_peaks[t]["hip"]
        knee_peak  = max(knee_peak,  trial_peaks[t]["knee"])  if not np.isnan(knee_peak)  else trial_peaks[t]["knee"]
        trunk_peak = max(trunk_peak, trial_peaks[t]["trunk"]) if not np.isnan(trunk_peak) else trial_peaks[t]["trunk"]
    else:
        hip_peak = knee_peak = trunk_peak = 0.0

    summary_rows.append({
        "trial": t,
        "label": TRIAL_LABELS[t],
        "start_frame": TRIAL_STARTS[t],
        "end_frame": TRIAL_ENDS[t],
        "start_time_s": round(TRIAL_STARTS[t] / fps, 2),
        "end_time_s": round(TRIAL_ENDS[t] / fps, 2),
        "peak_hip_flexion_deg": round(hip_peak, 1),
        "peak_knee_flexion_deg": round(knee_peak, 1),
        "peak_trunk_flexion_deg": round(trunk_peak, 1),
    })

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(os.path.join(OUTPUT_DIR, "summary_peaks_by_trial.csv"), index=False)

# ------------------------------
# 9. DASHBOARD
# ------------------------------
plt.style.use("seaborn-v0_8-whitegrid")
fig = plt.figure(figsize=(16, 12), facecolor="#f8f9fa")
gs = GridSpec(4, 2, figure=fig, height_ratios=[0.85, 1.0, 1.0, 1.1],
              hspace=0.40, wspace=0.28)

ax0 = fig.add_subplot(gs[0, :])
ax0.axis("off")
ax0.set_xlim(0, 1)
ax0.set_ylim(0, 1)
ax0.text(0.5, 0.92, "Put-on-Socks Test  -  Four Distinct Trials",
         ha="center", va="top", fontsize=18, fontweight="bold", color="#1a1a2e")
ax0.text(0.5, 0.68,
         "Hip/Knee flexion = 180 - joint angle (higher = more flexed)  |  Trunk = angle from vertical",
         ha="center", va="top", fontsize=11, color="#444")

colors = ["#e94560", "#0f3460", "#e94560", "#0f3460"]
card_w = 0.20
for i, row in enumerate(summary_rows):
    x = 0.06 + i * 0.24
    ax0.add_patch(plt.Rectangle((x, 0.08), card_w, 0.48, facecolor=colors[i], alpha=0.92))
    ax0.text(x + card_w/2, 0.48, f"Trial {row['trial']}", ha="center", va="center",
             fontsize=11, fontweight="bold", color="white")
    ax0.text(x + card_w/2, 0.36, row["label"], ha="center", va="center",
             fontsize=9, color="white")
    ax0.text(x + card_w/2, 0.20,
             f"Hip {row['peak_hip_flexion_deg']:.0f}  Knee {row['peak_knee_flexion_deg']:.0f}\nTrunk {row['peak_trunk_flexion_deg']:.0f} deg",
             ha="center", va="center", fontsize=9, color="white")

ax1 = fig.add_subplot(gs[1, 0])
ax1.plot(df["time_s"], df["hip_flexion_L"], label="Left", color="#e94560", lw=1.5, alpha=0.8)
ax1.plot(df["time_s"], df["hip_flexion_R"], label="Right", color="#0f3460", lw=1.5, alpha=0.8)
for t in range(1, 5):
    ax1.axvspan(TRIAL_STARTS[t]/fps, TRIAL_ENDS[t]/fps, alpha=0.12, color=colors[t-1])
    ax1.axvline(TRIAL_STARTS[t]/fps, color=colors[t-1], ls="--", lw=1, alpha=0.7)
ax1.set_ylabel("Hip Flexion (deg)")
ax1.set_title("Hip Flexion (shaded = trial windows)", fontweight="bold")
ax1.legend(loc="upper right", fontsize=8)
ax1.set_xlabel("Time (s)")

ax2 = fig.add_subplot(gs[1, 1])
ax2.plot(df["time_s"], df["knee_flexion_L"], label="Left", color="#e94560", lw=1.5, alpha=0.8)
ax2.plot(df["time_s"], df["knee_flexion_R"], label="Right", color="#0f3460", lw=1.5, alpha=0.8)
for t in range(1, 5):
    ax2.axvspan(TRIAL_STARTS[t]/fps, TRIAL_ENDS[t]/fps, alpha=0.12, color=colors[t-1])
    ax2.axvline(TRIAL_STARTS[t]/fps, color=colors[t-1], ls="--", lw=1, alpha=0.7)
ax2.set_ylabel("Knee Flexion (deg)")
ax2.set_title("Knee Flexion (shaded = trial windows)", fontweight="bold")
ax2.legend(loc="upper right", fontsize=8)
ax2.set_xlabel("Time (s)")

ax3 = fig.add_subplot(gs[2, :])
ax3.plot(df["time_s"], df["trunk_flexion"], color="#16213e", lw=1.8)
for t in range(1, 5):
    ax3.axvspan(TRIAL_STARTS[t]/fps, TRIAL_ENDS[t]/fps, alpha=0.12, color=colors[t-1])
    ax3.axvline(TRIAL_STARTS[t]/fps, color=colors[t-1], ls="--", lw=1.2, alpha=0.8)
ax3.set_ylabel("Trunk Flexion (deg)")
ax3.set_xlabel("Time (s)")
ax3.set_title("Trunk Flexion Angle (from vertical)", fontweight="bold")

ax4 = fig.add_subplot(gs[3, :])
ax4.axis("off")
table_data = [
    [f"Trial {r['trial']}", r["label"],
     f"{r['start_frame']}-{r['end_frame']}",
     f"{r['start_time_s']}-{r['end_time_s']}s",
     f"{r['peak_hip_flexion_deg']:.1f}",
     f"{r['peak_knee_flexion_deg']:.1f}",
     f"{r['peak_trunk_flexion_deg']:.1f}"]
    for r in summary_rows
]
col_labels = ["Trial", "Side / Rep", "Frames", "Time (s)", "Peak Hip", "Peak Knee", "Peak Trunk"]
table = ax4.table(cellText=table_data, colLabels=col_labels, loc="center", cellLoc="center")
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1.15, 1.6)
for (row, col), cell in table.get_celld().items():
    if row == 0:
        cell.set_facecolor("#1a1a2e")
        cell.set_text_props(color="white", fontweight="bold")
    else:
        cell.set_facecolor("#f0f0f0" if row % 2 == 0 else "white")

fig.text(0.5, 0.01,
         f"Source: {Path(VIDEO_PATH).name}  |  FPS: {fps:.1f}  |  Frames: {len(df)}  |  "
         f"Hip/Knee = 180 - joint angle  |  Trunk = from vertical",
         ha="center", fontsize=9, color="#666")

plt.savefig(DASHBOARD_PNG, dpi=180, bbox_inches="tight", facecolor=fig.get_facecolor())
plt.close()
print(f"Dashboard saved -> {DASHBOARD_PNG}")

# ------------------------------
# 10. CONSOLE
# ------------------------------
print("\n" + "="*70)
print("     PUT-ON-SOCKS TEST - RESULTS BY TRIAL")
print("="*70)
for r in summary_rows:
    print(f"  Trial {r['trial']}  {r['label']:14s}  "
          f"Hip {r['peak_hip_flexion_deg']:5.1f} deg  "
          f"Knee {r['peak_knee_flexion_deg']:5.1f} deg  "
          f"Trunk {r['peak_trunk_flexion_deg']:5.1f} deg  "
          f"[{r['start_frame']}-{r['end_frame']}]")
print("-"*70)
print(f"  Annotated video          : {OUTPUT_VIDEO}")
print(f"  Frame-level CSV          : {OUTPUT_CSV}")
print(f"  Summary by trial CSV     : {os.path.join(OUTPUT_DIR, 'summary_peaks_by_trial.csv')}")
print(f"  Results dashboard        : {DASHBOARD_PNG}")
print("="*70)

Model downloaded.
Trial windows (frames):
  Trial 1 (Right - Rep 1): 324 -> 501  (5.41s - 8.36s)
  Trial 2 (Right - Rep 2): 502 -> 648  (8.38s - 10.81s)
  Trial 3 (Left  - Rep 1): 649 -> 778  (10.83s - 12.98s)
  Trial 4 (Left  - Rep 2): 779 -> 927  (13.00s - 15.47s)


W0000 00:00:1787501928.718690     153 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1787501928.759640     153 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


Processed 30/928 frames...
Processed 60/928 frames...
Processed 90/928 frames...
Processed 120/928 frames...
Processed 150/928 frames...
Processed 180/928 frames...
Processed 210/928 frames...
Processed 240/928 frames...
Processed 270/928 frames...
Processed 300/928 frames...
Processed 330/928 frames...
Processed 360/928 frames...
Processed 390/928 frames...
Processed 420/928 frames...
Processed 450/928 frames...
Processed 480/928 frames...
Processed 510/928 frames...
Processed 540/928 frames...
Processed 570/928 frames...
Processed 600/928 frames...
Processed 630/928 frames...
Processed 660/928 frames...
Processed 690/928 frames...
Processed 720/928 frames...
Processed 750/928 frames...
Processed 780/928 frames...
Processed 810/928 frames...
Processed 840/928 frames...
Processed 870/928 frames...
Processed 900/928 frames...

Annotated video saved -> /kaggle/working/socks_results/put_on_socks_annotated.mp4
CSV saved -> /kaggle/working/socks_results/put_on_socks_metrics.csv
Dashboard sa

In [5]:
!pip install -q mediapipe opencv-python scipy pandas

import cv2
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
import numpy as np
import pandas as pd
from scipy.signal import savgol_filter, find_peaks
from pathlib import Path
import urllib.request
import os

# ----------------------------- USER CONFIG -----------------------------
VIDEO_PATH = "/kaggle/input/datasets/sarangsharma/4mwt-axspa/walking.mp4"
SUBJECT_HEIGHT_CM = 165.0          # actual subject height
FLOOR_DISTANCE_M  = 4.0            # measured floor distance (start → stop)
OUTPUT_DIR = Path("/kaggle/working/")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_VIDEO = OUTPUT_DIR / "4MWT_FINAL_annotated.mp4"
OUTPUT_CSV   = OUTPUT_DIR / "4MWT_FINAL_metrics.csv"
# -----------------------------------------------------------------------

MODEL_URL = "https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_lite/float16/1/pose_landmarker_lite.task"
MODEL_PATH = str(OUTPUT_DIR / "pose_landmarker_lite.task")

if not os.path.exists(MODEL_PATH):
    print("Downloading Pose Landmarker model...")
    urllib.request.urlretrieve(MODEL_URL, MODEL_PATH)
    print("Model downloaded.\n")

BaseOptions = python.BaseOptions
PoseLandmarker = vision.PoseLandmarker
PoseLandmarkerOptions = vision.PoseLandmarkerOptions
VisionRunningMode = vision.RunningMode

options = PoseLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=MODEL_PATH),
    running_mode=VisionRunningMode.VIDEO,
    num_poses=1,
    min_pose_detection_confidence=0.5,
    min_pose_presence_confidence=0.5,
    min_tracking_confidence=0.5
)

# ------------------------------------------------------------------
# Geometry helpers
# ------------------------------------------------------------------
def get_px(lm, w, h):
    return np.array([lm.x * w, lm.y * h], dtype=np.float64)

def calculate_angle(a, b, c):
    ba = a - b
    bc = c - b
    cosang = np.clip(np.dot(ba, bc) / (np.linalg.norm(ba) * np.linalg.norm(bc) + 1e-8), -1.0, 1.0)
    return float(np.degrees(np.arccos(cosang)))

def trunk_flexion_deg(shoulder, hip):
    dx = shoulder[0] - hip[0]
    dy = shoulder[1] - hip[1]
    return abs(float(np.degrees(np.arctan2(dx, -dy))))

def hip_flexion_deg(shoulder, hip, knee):
    raw = calculate_angle(shoulder, hip, knee)
    return max(0.0, 180.0 - raw)

# ------------------------------------------------------------------
# Main processing
# ------------------------------------------------------------------
def process_4mwt(video_path, height_cm, floor_distance_m):
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise FileNotFoundError(f"Cannot open video: {video_path}")

    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    w   = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h   = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    print(f"Video: {w}×{h} @ {fps:.1f} fps ({total_frames} frames)\n")

    times, trunk_raw, hip_raw = [], [], []
    hip_x, ankle_x, body_h_px = [], [], []

    with PoseLandmarker.create_from_options(options) as landmarker:
        idx = 0
        while True:
            ret, frame = cap.read()
            if not ret:
                break

            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            mp_img = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
            res = landmarker.detect_for_video(mp_img, int(idx * 1000 / fps))

            t = idx / fps
            trunk = hip = hx = ax = bh = np.nan

            if res.pose_landmarks:
                lms = res.pose_landmarks[0]

                r_vis = (lms[12].visibility + lms[24].visibility +
                         lms[26].visibility + lms[28].visibility) / 4.0
                l_vis = (lms[11].visibility + lms[23].visibility +
                         lms[25].visibility + lms[27].visibility) / 4.0

                if l_vis > r_vis + 0.08:
                    sh     = get_px(lms[11], w, h)
                    hip_pt = get_px(lms[23], w, h)
                    kn     = get_px(lms[25], w, h)
                    ank    = get_px(lms[27], w, h)
                    nose   = get_px(lms[0], w, h)
                else:
                    sh     = get_px(lms[12], w, h)
                    hip_pt = get_px(lms[24], w, h)
                    kn     = get_px(lms[26], w, h)
                    ank    = get_px(lms[28], w, h)
                    nose   = get_px(lms[0], w, h)

                trunk = trunk_flexion_deg(sh, hip_pt)
                hip   = hip_flexion_deg(sh, hip_pt, kn)
                hx    = hip_pt[0]
                ax    = ank[0]
                bh    = abs(ank[1] - nose[1])

            times.append(t)
            trunk_raw.append(trunk)
            hip_raw.append(hip)
            hip_x.append(hx)
            ankle_x.append(ax)
            body_h_px.append(bh)
            idx += 1

    cap.release()

    times     = np.asarray(times)
    trunk_raw = np.asarray(trunk_raw, dtype=float)
    hip_raw   = np.asarray(hip_raw, dtype=float)
    hip_x     = np.asarray(hip_x, dtype=float)
    ankle_x   = np.asarray(ankle_x, dtype=float)
    body_h_px = np.asarray(body_h_px, dtype=float)

    def fill_nan(a):
        n = np.isnan(a)
        if n.any() and (~n).any():
            a[n] = np.interp(np.flatnonzero(n), np.flatnonzero(~n), a[~n])
        return a

    trunk_raw = fill_nan(trunk_raw)
    hip_raw   = fill_nan(hip_raw)
    hip_x     = fill_nan(hip_x)
    ankle_x   = fill_nan(ankle_x)
    body_h_px = fill_nan(body_h_px)

    # Scale (still useful for stride length)
    median_body_px = np.nanmedian(body_h_px[body_h_px > 50])
    if median_body_px > 50:
        scale = (height_cm / 100.0) / median_body_px
        print(f"Scale factor: {scale*1000:.2f} mm/pixel\n")
    else:
        scale = np.nan
        print("Warning: unreliable scale factor\n")

    # ---------- Standstill detection ----------
    hip_disp = np.abs(np.gradient(hip_x))
    quiet_end = 6
    for i in range(4, min(30, len(hip_disp)-4)):
        if (np.mean(hip_disp[i:i+4]) > 1.8 * np.median(hip_disp[:8]) and
            np.mean(hip_disp[i:i+4]) > np.percentile(hip_disp[:20], 55)):
            quiet_end = i
            break
    quiet_end = max(5, min(quiet_end, 12))

    baseline_trunk = float(np.nanmedian(trunk_raw[:quiet_end]))
    baseline_hip   = float(np.nanmedian(hip_raw[:quiet_end]))

    print(f"Standstill: first {quiet_end} frames (~{quiet_end/fps:.3f}s)")
    print(f"  Baseline Trunk = {baseline_trunk:.1f}°   Baseline Hip = {baseline_hip:.1f}°\n")

    trunk = np.maximum(trunk_raw - baseline_trunk, 0.0)
    hip   = np.maximum(hip_raw   - baseline_hip,   0.0)

    win = max(5, int(0.25 * fps) | 1)
    trunk_s   = savgol_filter(trunk, win, 2)
    hip_s     = savgol_filter(hip,   win, 2)
    ang_vel_s = savgol_filter(np.gradient(hip_s, times), win, 2)

    trunk_s[:quiet_end]  = 0.0
    hip_s[:quiet_end]    = 0.0
    ang_vel_s[:quiet_end] = 0.0

    # ---------- Spatiotemporal parameters ----------
    walk_start = quiet_end
    walk_end   = len(times) - 1
    completion_time = float(times[walk_end] - times[walk_start])   # seconds

    # Accurate gait speed using measured 4 m floor distance
    gait_speed = floor_distance_m / completion_time if completion_time > 0.05 else np.nan

    # Step detection for cadence & stride length
    ankle_s = savgol_filter(ankle_x, win, 2)
    ankle_vel = np.gradient(ankle_s, times)
    min_dist = int(0.30 * fps)
    peaks, _ = find_peaks(np.abs(ankle_vel[walk_start:]), 
                          distance=min_dist, 
                          prominence=np.std(ankle_vel)*0.35)
    peaks = peaks + walk_start
    n_steps = len(peaks)

    # Cadence (steps per minute)
    if completion_time > 0.1 and n_steps >= 2:
        cadence = (n_steps / completion_time) * 60.0
    else:
        cadence = np.nan

    # Mean stride length (using scale)
    stride_lengths = []
    if n_steps >= 2 and not np.isnan(scale):
        for i in range(n_steps-1):
            dx_px = abs(ankle_s[peaks[i+1]] - ankle_s[peaks[i]])
            stride_lengths.append(dx_px * scale)
        mean_stride = float(np.mean(stride_lengths))
    else:
        mean_stride = np.nan

    # Kinematic peaks
    peak_trunk = float(np.nanmax(trunk_s[quiet_end:]))
    peak_hip   = float(np.nanmax(hip_s[quiet_end:]))
    peak_vel   = float(np.nanmax(np.abs(ang_vel_s[quiet_end:])))

    # -------------------- Results Summary --------------------
    print("=" * 70)
    print(" 4MWT – RESULTS SUMMARY (Camera-based, Lateral View)")
    print("=" * 70)
    print(f" Completion Time             : {completion_time:6.2f}  seconds")
    print(f" Gait Speed                  : {gait_speed:6.2f}  m/s")
    print(f" Mean Stride Length          : {mean_stride:6.2f}  m" if not np.isnan(mean_stride) else " Mean Stride Length          :   N/A")
    print(f" Cadence                     : {cadence:6.1f}  steps/min" if not np.isnan(cadence) else " Cadence                     :   N/A")
    print(f" Peak Trunk Flexion          : {peak_trunk:6.1f}  degrees")
    print(f" Peak Hip Flexion Angle      : {peak_hip:6.1f}  degrees")
    print(f" Peak Angular Velocity       : {peak_vel:6.1f}  deg/s")
    print("=" * 70)
    print()

    # -------------------- CSV --------------------
    df = pd.DataFrame({
        "frame": np.arange(len(times)),
        "time_s": np.round(times, 3),
        "trunk_flexion_deg": np.round(trunk_s, 1),
        "hip_flexion_deg": np.round(hip_s, 1),
        "hip_angular_velocity_deg_s": np.round(ang_vel_s, 1)
    })

    with open(OUTPUT_CSV, "w") as f:
        f.write("# 4MWT – Per-frame kinematic data\n")
        f.write("# Units: time=s, flexion=deg, angular velocity=deg/s\n")
        f.write(f"# Subject Height      = {height_cm} cm\n")
        f.write(f"# Floor Distance      = {floor_distance_m} m\n")
        f.write(f"# Completion Time     = {completion_time:.2f} s\n")
        f.write(f"# Gait Speed          = {gait_speed:.2f} m/s\n")
        f.write(f"# Mean Stride Length  = {mean_stride:.2f} m\n" if not np.isnan(mean_stride) else "# Mean Stride Length = N/A\n")
        f.write(f"# Cadence             = {cadence:.1f} steps/min\n" if not np.isnan(cadence) else "# Cadence = N/A\n")
        f.write(f"# Peak Trunk Flexion  = {peak_trunk:.1f} deg\n")
        f.write(f"# Peak Hip Flexion    = {peak_hip:.1f} deg\n")
        f.write(f"# Peak Ang. Velocity  = {peak_vel:.1f} deg/s\n\n")

    df.to_csv(OUTPUT_CSV, mode="a", index=False)

    summary_df = pd.DataFrame([
        {"Metric": "Completion Time", "Value": round(completion_time, 2), "Unit": "seconds"},
        {"Metric": "Gait Speed", "Value": round(gait_speed, 2), "Unit": "m/s"},
        {"Metric": "Mean Stride Length", "Value": round(mean_stride, 2) if not np.isnan(mean_stride) else None, "Unit": "m"},
        {"Metric": "Cadence", "Value": round(cadence, 1) if not np.isnan(cadence) else None, "Unit": "steps/min"},
        {"Metric": "Peak Trunk Flexion", "Value": round(peak_trunk, 1), "Unit": "degrees"},
        {"Metric": "Peak Hip Flexion Angle", "Value": round(peak_hip, 1), "Unit": "degrees"},
        {"Metric": "Peak Angular Velocity", "Value": round(peak_vel, 1), "Unit": "degrees/second"}
    ])
    with open(OUTPUT_CSV, "a") as f:
        f.write("\n# Results Summary\n")
    summary_df.to_csv(OUTPUT_CSV, mode="a", index=False)

    print(f"CSV saved → {OUTPUT_CSV}")

    # -------------------- Annotated Video --------------------
    cap = cv2.VideoCapture(str(video_path))
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    out = cv2.VideoWriter(str(OUTPUT_VIDEO), fourcc, fps, (w, h))

    with PoseLandmarker.create_from_options(options) as landmarker:
        idx = 0
        while True:
            ret, frame = cap.read()
            if not ret:
                break

            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            mp_img = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
            res = landmarker.detect_for_video(mp_img, int(idx * 1000 / fps))

            if res.pose_landmarks:
                for i in [11, 12, 23, 24, 25, 26, 27, 28]:
                    x = int(res.pose_landmarks[0][i].x * w)
                    y = int(res.pose_landmarks[0][i].y * h)
                    cv2.circle(frame, (x, y), 5, (0, 255, 120), -1)

            overlay = frame.copy()
            cv2.rectangle(overlay, (8, 8), (560, 230), (15, 15, 15), -1)
            cv2.addWeighted(overlay, 0.65, frame, 0.35, 0, frame)
            cv2.rectangle(frame, (8, 8), (560, 230), (0, 220, 255), 2)

            t_cur   = times[idx] if idx < len(times) else 0
            trunk_v = trunk_s[idx] if idx < len(trunk_s) else 0
            hip_v   = hip_s[idx] if idx < len(hip_s) else 0
            vel_v   = abs(ang_vel_s[idx]) if idx < len(ang_vel_s) else 0

            font = cv2.FONT_HERSHEY_SIMPLEX
            y0, dy = 30, 25
            cv2.putText(frame, "4MWT  Lateral View", (18, y0), font, 0.58, (0, 255, 255), 2)
            cv2.putText(frame, f"Time: {t_cur:5.2f} s", (18, y0+dy), font, 0.55, (255,255,255), 2)
            cv2.putText(frame, f"Trunk Flex : {trunk_v:5.1f} deg", (18, y0+2*dy), font, 0.55, (80,255,160), 2)
            cv2.putText(frame, f"Hip Flex   : {hip_v:5.1f} deg", (18, y0+3*dy), font, 0.55, (80,200,255), 2)
            cv2.putText(frame, f"Ang. Vel   : {vel_v:5.1f} deg/s", (18, y0+4*dy), font, 0.55, (0,180,255), 2)

            # Summary metrics (shown in later part of video)
            if idx > len(times) * 0.55:
                cv2.putText(frame, f"Speed : {gait_speed:.2f} m/s", (18, y0+5*dy), font, 0.52, (200,200,255), 1)
                cv2.putText(frame, f"Stride: {mean_stride:.2f} m" if not np.isnan(mean_stride) else "Stride: N/A",
                            (18, y0+6*dy), font, 0.52, (200,200,255), 1)
                cv2.putText(frame, f"Cadence: {cadence:.0f} steps/min" if not np.isnan(cadence) else "Cadence: N/A",
                            (18, y0+7*dy), font, 0.52, (200,200,255), 1)

            out.write(frame)
            idx += 1

    cap.release()
    out.release()
    print(f"Annotated video saved → {OUTPUT_VIDEO}\n")

    return summary_df

# ===================== RUN =====================
summary = process_4mwt(VIDEO_PATH, SUBJECT_HEIGHT_CM, FLOOR_DISTANCE_M)

Video: 832×464 @ 59.9 fps (197 frames)



W0000 00:00:1787501962.618873     172 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1787501962.649370     171 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


Scale factor: 7.36 mm/pixel

Standstill: first 8 frames (~0.133s)
  Baseline Trunk = 5.5°   Baseline Hip = 6.4°

 4MWT – RESULTS SUMMARY (Camera-based, Lateral View)
 Completion Time             :   3.14  seconds
 Gait Speed                  :   1.28  m/s
 Mean Stride Length          :   1.42  m
 Cadence                     :   76.5  steps/min
 Peak Trunk Flexion          :    8.1  degrees
 Peak Hip Flexion Angle      :   29.0  degrees
 Peak Angular Velocity       :  165.3  deg/s

CSV saved → /kaggle/working/4MWT_FINAL_metrics.csv


W0000 00:00:1787501966.672057     184 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1787501966.702244     186 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


Annotated video saved → /kaggle/working/4MWT_FINAL_annotated.mp4



In [6]:
!pip install -q mediapipe opencv-python scipy pandas

import cv2
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
import numpy as np
import pandas as pd
from scipy.signal import savgol_filter, find_peaks, medfilt
from pathlib import Path
import urllib.request
import os

# ----------------------------- USER CONFIG -----------------------------
VIDEO_PATH = "/kaggle/input/datasets/sarangsharma/bending-axspa/bending.mp4"
OUTPUT_DIR = Path("/kaggle/working/")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_VIDEO = OUTPUT_DIR / "Bending_Task_HipFocus_annotated.mp4"
OUTPUT_CSV   = OUTPUT_DIR / "Bending_Task_HipFocus_metrics.csv"
# -----------------------------------------------------------------------

MODEL_URL = "https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_lite/float16/1/pose_landmarker_lite.task"
MODEL_PATH = str(OUTPUT_DIR / "pose_landmarker_lite.task")

if not os.path.exists(MODEL_PATH):
    print("Downloading Pose Landmarker model...")
    urllib.request.urlretrieve(MODEL_URL, MODEL_PATH)
    print("Model downloaded.\n")

BaseOptions = python.BaseOptions
PoseLandmarker = vision.PoseLandmarker
PoseLandmarkerOptions = vision.PoseLandmarkerOptions
VisionRunningMode = vision.RunningMode

options = PoseLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=MODEL_PATH),
    running_mode=VisionRunningMode.VIDEO,
    num_poses=1,
    min_pose_detection_confidence=0.5,
    min_pose_presence_confidence=0.5,
    min_tracking_confidence=0.5
)

def get_px(lm, w, h):
    return np.array([lm.x * w, lm.y * h], dtype=np.float64)

def trunk_flexion_deg(shoulder, hip):
    dx = shoulder[0] - hip[0]
    dy = shoulder[1] - hip[1]
    return abs(float(np.degrees(np.arctan2(dx, -dy))))

def hip_flexion_deg(shoulder, hip, knee):
    ba = shoulder - hip
    bc = knee - hip
    cosang = np.clip(np.dot(ba, bc) / (np.linalg.norm(ba) * np.linalg.norm(bc) + 1e-8), -1.0, 1.0)
    raw = float(np.degrees(np.arccos(cosang)))
    return max(0.0, 180.0 - raw)

def fill_nan_1d(a):
    n = np.isnan(a)
    if n.any() and (~n).any():
        a = a.copy()
        a[n] = np.interp(np.flatnonzero(n), np.flatnonzero(~n), a[~n])
    return a

def smooth_coords(coords, fps):
    if coords.shape[0] < 5:
        return coords
    k = max(5, int(0.15 * fps) | 1)
    x = medfilt(coords[:, 0], kernel_size=k)
    y = medfilt(coords[:, 1], kernel_size=k)
    win = max(5, int(0.20 * fps) | 1)
    if win >= len(x):
        win = len(x) - 1 if len(x) % 2 == 0 else len(x)
        win = max(3, win)
    x = savgol_filter(x, win, 2)
    y = savgol_filter(y, win, 2)
    return np.column_stack([x, y])

def process_bending_task(video_path):
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise FileNotFoundError(f"Cannot open video: {video_path}")

    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    print(f"Video: {w}×{h} @ {fps:.1f} fps ({total_frames} frames)\n")

    times, shoulder_list, hip_list, knee_list, hip_x_motion = [], [], [], [], []

    with PoseLandmarker.create_from_options(options) as landmarker:
        idx = 0
        while True:
            ret, frame = cap.read()
            if not ret:
                break

            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            mp_img = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
            res = landmarker.detect_for_video(mp_img, int(idx * 1000 / fps))

            t = idx / fps
            sh = hip_pt = kn = np.array([np.nan, np.nan])
            hx = np.nan

            if res.pose_landmarks:
                lms = res.pose_landmarks[0]
                r_vis = (lms[12].visibility + lms[24].visibility + lms[26].visibility) / 3.0
                l_vis = (lms[11].visibility + lms[23].visibility + lms[25].visibility) / 3.0

                if l_vis > r_vis + 0.08:
                    sh = get_px(lms[11], w, h)
                    hip_pt = get_px(lms[23], w, h)
                    kn = get_px(lms[25], w, h)
                else:
                    sh = get_px(lms[12], w, h)
                    hip_pt = get_px(lms[24], w, h)
                    kn = get_px(lms[26], w, h)
                hx = hip_pt[0]

            times.append(t)
            shoulder_list.append(sh)
            hip_list.append(hip_pt)
            knee_list.append(kn)
            hip_x_motion.append(hx)
            idx += 1

    cap.release()

    times = np.asarray(times)
    shoulder = np.asarray(shoulder_list, dtype=float)
    hip      = np.asarray(hip_list, dtype=float)
    knee     = np.asarray(knee_list, dtype=float)
    hip_x    = np.asarray(hip_x_motion, dtype=float)

    for arr in (shoulder, hip, knee):
        arr[:, 0] = fill_nan_1d(arr[:, 0])
        arr[:, 1] = fill_nan_1d(arr[:, 1])
    hip_x = fill_nan_1d(hip_x)

    print("Smoothing landmark coordinates...")
    shoulder_s = smooth_coords(shoulder, fps)
    hip_s_coord = smooth_coords(hip, fps)
    knee_s = smooth_coords(knee, fps)

    # Angles
    trunk_raw = np.array([trunk_flexion_deg(shoulder_s[i], hip_s_coord[i]) for i in range(len(times))])
    hip_raw   = np.array([hip_flexion_deg(shoulder_s[i], hip_s_coord[i], knee_s[i]) for i in range(len(times))])

    # Baseline
    hip_disp = np.abs(np.gradient(hip_x))
    quiet_end = 6
    for i in range(4, min(30, len(hip_disp)-4)):
        if (np.mean(hip_disp[i:i+4]) > 1.8 * np.median(hip_disp[:8]) and
            np.mean(hip_disp[i:i+4]) > np.percentile(hip_disp[:20], 55)):
            quiet_end = i
            break
    quiet_end = max(5, min(quiet_end, 15))

    baseline_trunk = float(np.nanmedian(trunk_raw[:quiet_end]))
    baseline_hip   = float(np.nanmedian(hip_raw[:quiet_end]))
    print(f"Standstill: first {quiet_end} frames (~{quiet_end/fps:.3f} s)")
    print(f"  Baseline Hip = {baseline_hip:.1f}°\n")

    trunk = np.maximum(trunk_raw - baseline_trunk, 0.0)
    hip   = np.maximum(hip_raw   - baseline_hip,   0.0)

    win = max(7, int(0.22 * fps) | 1)
    trunk_s = savgol_filter(trunk, win, 2)
    hip_s   = savgol_filter(hip, win, 2)
    trunk_s[:quiet_end] = 0.0
    hip_s[:quiet_end]   = 0.0

    # Angular velocity (from trunk)
    ang_vel = np.gradient(trunk_s, times)
    ang_vel = np.clip(ang_vel, -180, 180)
    ang_vel_s = savgol_filter(ang_vel, win, 2)
    ang_vel_s[:quiet_end] = 0.0

    # ----- Metrics focused on Hip + Time + Angular Velocity -----
    post = slice(quiet_end, None)

    # Peak Hip Flexion (dual peak if possible)
    k_hip = max(7, int(0.20 * fps) | 1)
    hip_loc = medfilt(hip_s, kernel_size=k_hip)

    min_dist = max(12, int(0.60 * fps))
    peaks_rel, _ = find_peaks(
        hip_loc[post],
        height=15.0,
        distance=min_dist,
        prominence=8.0
    )

    if len(peaks_rel) >= 2:
        heights = hip_loc[post][peaks_rel]
        top2 = np.argsort(heights)[-2:]
        peaks_rel = np.sort(peaks_rel[top2])
    elif len(peaks_rel) == 0:
        peaks_rel = np.array([np.argmax(hip_loc[post])])

    peak_indices = peaks_rel + quiet_end
    peak_hip_values = hip_s[peak_indices]
    peak_hip_times  = times[peak_indices]

    overall_peak_hip = float(np.max(peak_hip_values))
    overall_peak_hip_time = float(peak_hip_times[np.argmax(peak_hip_values)])

    peak_ang_velocity = float(np.nanmax(np.abs(ang_vel_s[post])))
    total_time = float(times[-1]) if len(times) else 0.0

    # -------------------- Results Summary --------------------
    print("=" * 70)
    print("  BENDING TASK (Pick-up Pen × 2) – RESULTS SUMMARY")
    print("  Focus: Time | Hip Flexion | Peak Angular Velocity")
    print("=" * 70)
    print(f"  Total Time                     : {total_time:6.2f} seconds")
    print()
    if len(peak_indices) >= 1:
        print(f"  Peak Hip Flexion (Rep 1)       : {peak_hip_values[0]:6.1f} degrees   @ {peak_hip_times[0]:5.2f} s")
    if len(peak_indices) >= 2:
        print(f"  Peak Hip Flexion (Rep 2)       : {peak_hip_values[1]:6.1f} degrees   @ {peak_hip_times[1]:5.2f} s")
    print(f"  Overall Peak Hip Flexion       : {overall_peak_hip:6.1f} degrees   @ {overall_peak_hip_time:5.2f} s")
    print(f"  Peak Angular Velocity          : {peak_ang_velocity:6.1f} degrees/second")
    print("=" * 70)
    print()

    # CSV
    df = pd.DataFrame({
        "frame": np.arange(len(times)),
        "time_s": np.round(times, 3),
        "hip_flexion_deg": np.round(hip_s, 1),
        "trunk_angular_velocity_deg_s": np.round(ang_vel_s, 1)
    })

    with open(OUTPUT_CSV, "w") as f:
        f.write("# Bending Task (Pick-up Pen × 2) – Hip Focus version\n")
        f.write("# Primary metrics: Time, Hip Flexion, Peak Angular Velocity\n")
        f.write("# Units: time = seconds, hip flexion = degrees, angular velocity = degrees/second\n")
        f.write(f"# Total Time = {total_time:.2f} s\n")
        if len(peak_indices) >= 1:
            f.write(f"# Peak Hip Flexion Rep1 = {peak_hip_values[0]:.1f} deg @ {peak_hip_times[0]:.2f} s\n")
        if len(peak_indices) >= 2:
            f.write(f"# Peak Hip Flexion Rep2 = {peak_hip_values[1]:.1f} deg @ {peak_hip_times[1]:.2f} s\n")
        f.write(f"# Overall Peak Hip Flexion = {overall_peak_hip:.1f} deg\n")
        f.write(f"# Peak Angular Velocity = {peak_ang_velocity:.1f} deg/s\n\n")

    df.to_csv(OUTPUT_CSV, mode="a", index=False)

    rows = [
        {"Metric": "Total Time", "Value": round(total_time, 2), "Unit": "seconds"},
    ]
    if len(peak_indices) >= 1:
        rows.append({"Metric": "Peak Hip Flexion (Rep 1)", "Value": round(float(peak_hip_values[0]), 1), "Unit": "degrees"})
        rows.append({"Metric": "Time of Peak (Rep 1)", "Value": round(float(peak_hip_times[0]), 2), "Unit": "seconds"})
    if len(peak_indices) >= 2:
        rows.append({"Metric": "Peak Hip Flexion (Rep 2)", "Value": round(float(peak_hip_values[1]), 1), "Unit": "degrees"})
        rows.append({"Metric": "Time of Peak (Rep 2)", "Value": round(float(peak_hip_times[1]), 2), "Unit": "seconds"})
    rows.append({"Metric": "Overall Peak Hip Flexion", "Value": round(overall_peak_hip, 1), "Unit": "degrees"})
    rows.append({"Metric": "Peak Angular Velocity", "Value": round(peak_ang_velocity, 1), "Unit": "degrees/second"})

    summary_df = pd.DataFrame(rows)
    with open(OUTPUT_CSV, "a") as f:
        f.write("\n# Results Summary\n")
    summary_df.to_csv(OUTPUT_CSV, mode="a", index=False)

    print(f"CSV saved → {OUTPUT_CSV}")

    # Annotated Video – shows Time, Hip Flexion, Angular Velocity
    cap = cv2.VideoCapture(str(video_path))
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    out = cv2.VideoWriter(str(OUTPUT_VIDEO), fourcc, fps, (w, h))

    with PoseLandmarker.create_from_options(options) as landmarker:
        idx = 0
        while True:
            ret, frame = cap.read()
            if not ret:
                break

            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            mp_img = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
            res = landmarker.detect_for_video(mp_img, int(idx * 1000 / fps))

            if res.pose_landmarks:
                for i in [11, 12, 23, 24, 25, 26]:
                    x = int(res.pose_landmarks[0][i].x * w)
                    y = int(res.pose_landmarks[0][i].y * h)
                    cv2.circle(frame, (x, y), 6, (0, 255, 120), -1)

            overlay = frame.copy()
            cv2.rectangle(overlay, (8, 8), (520, 175), (15, 15, 15), -1)
            cv2.addWeighted(overlay, 0.65, frame, 0.35, 0, frame)
            cv2.rectangle(frame, (8, 8), (520, 175), (0, 220, 255), 2)

            t_cur = times[idx] if idx < len(times) else 0
            hip_v = hip_s[idx] if idx < len(hip_s) else 0
            vel_v = abs(ang_vel_s[idx]) if idx < len(ang_vel_s) else 0

            font = cv2.FONT_HERSHEY_SIMPLEX
            y0, dy = 38, 32

            cv2.putText(frame, "Bending Task – Hip Focus", (16, y0),
                        font, 0.62, (0, 255, 255), 2)
            cv2.putText(frame, f"Time: {t_cur:5.2f} s", (16, y0 + dy),
                        font, 0.65, (255, 255, 255), 2)
            cv2.putText(frame, f"Hip Flexion : {hip_v:5.1f} deg", (16, y0 + 2*dy),
                        font, 0.65, (80, 200, 255), 2)
            cv2.putText(frame, f"Ang. Velocity : {vel_v:5.1f} deg/s", (16, y0 + 3*dy),
                        font, 0.65, (0, 180, 255), 2)

            if idx in peak_indices:
                cv2.putText(frame, "PEAK", (430, 50), font, 0.7, (0, 0, 255), 2)

            out.write(frame)
            idx += 1

    cap.release()
    out.release()
    print(f"Annotated video saved → {OUTPUT_VIDEO}\n")

    return summary_df

# ===================== RUN =====================
summary = process_bending_task(VIDEO_PATH)
print(summary)

Video: 576×1024 @ 30.0 fps (121 frames)



W0000 00:00:1787501974.641242     203 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1787501974.671987     204 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


Smoothing landmark coordinates...
Standstill: first 10 frames (~0.333 s)
  Baseline Hip = 7.6°

  BENDING TASK (Pick-up Pen × 2) – RESULTS SUMMARY
  Focus: Time | Hip Flexion | Peak Angular Velocity
  Total Time                     :   4.00 seconds

  Peak Hip Flexion (Rep 1)       :  133.9 degrees   @  1.03 s
  Peak Hip Flexion (Rep 2)       :  132.4 degrees   @  3.00 s
  Overall Peak Hip Flexion       :  133.9 degrees   @  1.03 s
  Peak Angular Velocity          :  184.8 degrees/second

CSV saved → /kaggle/working/Bending_Task_HipFocus_metrics.csv


W0000 00:00:1787501977.205379     215 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1787501977.230126     217 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


Annotated video saved → /kaggle/working/Bending_Task_HipFocus_annotated.mp4

                     Metric   Value            Unit
0                Total Time    4.00         seconds
1  Peak Hip Flexion (Rep 1)  133.90         degrees
2      Time of Peak (Rep 1)    1.03         seconds
3  Peak Hip Flexion (Rep 2)  132.40         degrees
4      Time of Peak (Rep 2)    3.00         seconds
5  Overall Peak Hip Flexion  133.90         degrees
6     Peak Angular Velocity  184.80  degrees/second


In [7]:
!pip install -q mediapipe opencv-python scipy pandas

import cv2
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
import numpy as np
import pandas as pd
from scipy.signal import savgol_filter, find_peaks, medfilt
from pathlib import Path
import urllib.request
import os
import warnings
warnings.filterwarnings("ignore")

# ----------------------------- USER CONFIG -----------------------------
VIDEO_PATH = "/kaggle/input/datasets/sarangsharma/cervical-axspa/cervicalmp4.mp4"  # <-- change to your path
OUTPUT_DIR = Path("/kaggle/working/")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_VIDEO = OUTPUT_DIR / "Cervical_Rotation_annotated.mp4"
OUTPUT_CSV = OUTPUT_DIR / "Cervical_Rotation_metrics.csv"
# -----------------------------------------------------------------------

MODEL_URL = "https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_lite/float16/1/pose_landmarker_lite.task"
MODEL_PATH = str(OUTPUT_DIR / "pose_landmarker_lite.task")
if not os.path.exists(MODEL_PATH):
    print("Downloading Pose Landmarker model...")
    urllib.request.urlretrieve(MODEL_URL, MODEL_PATH)
    print("Model downloaded.\n")

BaseOptions = python.BaseOptions
PoseLandmarker = vision.PoseLandmarker
PoseLandmarkerOptions = vision.PoseLandmarkerOptions
VisionRunningMode = vision.RunningMode

options = PoseLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=MODEL_PATH),
    running_mode=VisionRunningMode.VIDEO,
    num_poses=1,
    min_pose_detection_confidence=0.5,
    min_pose_presence_confidence=0.5,
    min_tracking_confidence=0.5
)

def get_px(lm, w, h):
    return np.array([lm.x * w, lm.y * h], dtype=np.float64)

def fill_nan_1d(a):
    a = np.asarray(a, dtype=float)
    n = np.isnan(a)
    if n.any() and (~n).any():
        a = a.copy()
        a[n] = np.interp(np.flatnonzero(n), np.flatnonzero(~n), a[~n])
    return a

def smooth_series(y, fps, med_k=None, sav_win=None):
    """Median + Savitzky-Golay. Windows tuned for ~60 fps smartphone video."""
    y = np.asarray(y, dtype=float)
    if len(y) < 5:
        return y
    if med_k is None:
        med_k = max(7, int(0.12 * fps) | 1)
    if med_k % 2 == 0:
        med_k += 1
    k = min(med_k, len(y) if len(y) % 2 == 1 else len(y) - 1)
    k = max(3, k)
    y = medfilt(y, kernel_size=k)
    if sav_win is None:
        sav_win = max(11, int(0.25 * fps) | 1)
    if sav_win >= len(y):
        sav_win = len(y) - 1 if len(y) % 2 == 0 else len(y)
        sav_win = max(5, sav_win)
    if sav_win % 2 == 0:
        sav_win -= 1
    try:
        y = savgol_filter(y, sav_win, 2)
    except Exception:
        pass
    return y

def cervical_yaw_deg(nose, left_sh, right_sh, scale_deg_per_px=1.55):
    """
    Approximate cervical rotation (yaw) – frontal smartphone view.
    Positive = subject LEFT (nose moves toward image-right).
    Negative = subject RIGHT.
    Uses horizontal offset of nose relative to stable mid-shoulder.
    Linear scale maps typical max offset (~40 px) → ~60-65°.
    Practical 2D projection estimate; validate vs goniometer for absolute values.
    """
    if (np.any(np.isnan(nose)) or np.any(np.isnan(left_sh)) or np.any(np.isnan(right_sh))):
        return np.nan
    mid_sh_x = 0.5 * (left_sh[0] + right_sh[0])
    offset = nose[0] - mid_sh_x
    return float(offset * scale_deg_per_px)

def process_cervical_rotation(video_path):
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise FileNotFoundError(f"Cannot open video: {video_path}")

    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    print(f"Video: {w}×{h} @ {fps:.1f} fps ({total_frames} frames)\n")

    times = []
    nose_list, lear_list, rear_list = [], [], []
    sh_l_list, sh_r_list = [], []

    with PoseLandmarker.create_from_options(options) as landmarker:
        idx = 0
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            mp_img = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
            res = landmarker.detect_for_video(mp_img, int(idx * 1000 / fps))

            t = idx / fps
            nose = lear = rear = shl = shr = np.array([np.nan, np.nan])

            if res.pose_landmarks and len(res.pose_landmarks) > 0:
                lms = res.pose_landmarks[0]
                def vis(i):
                    try:
                        return float(lms[i].visibility)
                    except Exception:
                        return 0.0

                if vis(0) > 0.4:
                    nose = get_px(lms[0], w, h)
                if vis(7) > 0.4:
                    lear = get_px(lms[7], w, h)
                if vis(8) > 0.4:
                    rear = get_px(lms[8], w, h)
                if vis(11) > 0.35:
                    shl = get_px(lms[11], w, h)
                if vis(12) > 0.35:
                    shr = get_px(lms[12], w, h)

            times.append(t)
            nose_list.append(nose)
            lear_list.append(lear)
            rear_list.append(rear)
            sh_l_list.append(shl)
            sh_r_list.append(shr)
            idx += 1

    cap.release()

    times = np.asarray(times, dtype=float)
    nose_arr = np.asarray(nose_list, dtype=float)
    lear_arr = np.asarray(lear_list, dtype=float)
    rear_arr = np.asarray(rear_list, dtype=float)
    sh_l_arr = np.asarray(sh_l_list, dtype=float)
    sh_r_arr = np.asarray(sh_r_list, dtype=float)

    for arr in (nose_arr, lear_arr, rear_arr, sh_l_arr, sh_r_arr):
        arr[:, 0] = fill_nan_1d(arr[:, 0])
        arr[:, 1] = fill_nan_1d(arr[:, 1])

    # ---------- Calibration (first 4 seconds) ----------
    calib_frames = max(5, int(4.0 * fps))
    calib_frames = min(calib_frames, max(30, len(times) // 4))
    print(f"Calibration window: first {calib_frames} frames (~{calib_frames/fps:.2f} s)")

    SCALE = 1.55  # deg/px – typical max offset ~40 px → ~60-65°
    yaw_list = [cervical_yaw_deg(nose_arr[i], sh_l_arr[i], sh_r_arr[i], scale_deg_per_px=SCALE)
                for i in range(len(times))]
    yaw_raw = fill_nan_1d(np.asarray(yaw_list, dtype=float))

    baseline = float(np.nanmedian(yaw_raw[:calib_frames]))
    print(f"Baseline yaw (median): {baseline:.2f}°  (will be subtracted)")
    yaw = yaw_raw - baseline

    print("Smoothing yaw series...")
    yaw_s = smooth_series(yaw, fps)
    yaw_s[:calib_frames] = 0.0

    # Angular velocity
    ang_vel = np.gradient(yaw_s, times)
    ang_vel = np.clip(ang_vel, -300, 300)
    ang_vel_s = smooth_series(ang_vel, fps,
                              med_k=max(9, int(0.15 * fps) | 1),
                              sav_win=max(15, int(0.30 * fps) | 1))
    ang_vel_s[:calib_frames] = 0.0

    # ---------- Metrics ----------
    post = slice(calib_frames, None)
    post_yaw = yaw_s[post]
    post_t = times[post]
    post_vel = ang_vel_s[post]

    peak_L = float(np.nanmax(post_yaw))
    peak_L_idx_rel = int(np.nanargmax(post_yaw))
    peak_L_time = float(post_t[peak_L_idx_rel])
    peak_L_frame = peak_L_idx_rel + calib_frames

    peak_R = float(-np.nanmin(post_yaw))
    peak_R_idx_rel = int(np.nanargmin(post_yaw))
    peak_R_time = float(post_t[peak_R_idx_rel])
    peak_R_frame = peak_R_idx_rel + calib_frames

    peak_ang_vel = float(np.nanmax(np.abs(post_vel)))
    peak_vel_idx_rel = int(np.nanargmax(np.abs(post_vel)))
    peak_vel_time = float(post_t[peak_vel_idx_rel])

    total_time = float(times[-1]) if len(times) else 0.0

    min_dist = max(int(0.4 * fps), 8)
    peaks_L_rel, _ = find_peaks(post_yaw, height=8.0, distance=min_dist, prominence=5.0)
    peaks_R_rel, _ = find_peaks(-post_yaw, height=8.0, distance=min_dist, prominence=5.0)

    # -------------------- Results Summary --------------------
    print("=" * 70)
    print(" CERVICAL ROTATION (Frontal Smartphone + MediaPipe Pose)")
    print(" Metrics: Peak L / R Rotation Angle | Peak Angular Velocity")
    print("=" * 70)
    print(f" Total Time                    : {total_time:6.2f} s")
    print(f" Calibration window            : {calib_frames/fps:6.2f} s")
    print()
    print(f" Peak Cervical Rotation LEFT   : {peak_L:6.1f} °  @ {peak_L_time:5.2f} s")
    print(f" Peak Cervical Rotation RIGHT  : {peak_R:6.1f} °  @ {peak_R_time:5.2f} s")
    print(f" Peak Angular Velocity         : {peak_ang_vel:6.1f} °/s  @ {peak_vel_time:5.2f} s")
    print()
    print(f" Detected Left peaks  (n={len(peaks_L_rel)}): {[round(float(post_yaw[i]),1) for i in peaks_L_rel]}")
    print(f" Detected Right peaks (n={len(peaks_R_rel)}): {[round(float(-post_yaw[i]),1) for i in peaks_R_rel]}")
    print("=" * 70)
    print()

    # -------------------- CSV --------------------
    df = pd.DataFrame({
        "frame": np.arange(len(times)),
        "time_s": np.round(times, 3),
        "cervical_rotation_deg": np.round(yaw_s, 2),   # + = Left, - = Right
        "angular_velocity_deg_s": np.round(ang_vel_s, 1)
    })

    with open(OUTPUT_CSV, "w") as f:
        f.write("# Cervical Rotation Assessment – Frontal Smartphone View + MediaPipe Pose\n")
        f.write("# Convention: positive angle = subject LEFT rotation; negative = subject RIGHT\n")
        f.write("# Initial ~4 s used for baseline calibration (forced to 0 after smoothing)\n")
        f.write("# Units: time = s, rotation = degrees, angular velocity = degrees/second\n")
        f.write(f"# Total Time = {total_time:.2f} s\n")
        f.write(f"# Peak Left  = {peak_L:.1f} deg @ {peak_L_time:.2f} s\n")
        f.write(f"# Peak Right = {peak_R:.1f} deg @ {peak_R_time:.2f} s\n")
        f.write(f"# Peak Angular Velocity = {peak_ang_vel:.1f} deg/s\n\n")

    df.to_csv(OUTPUT_CSV, mode="a", index=False)

    summary_rows = [
        {"Metric": "Total Time", "Value": round(total_time, 2), "Unit": "seconds"},
        {"Metric": "Peak Cervical Rotation LEFT", "Value": round(peak_L, 1), "Unit": "degrees"},
        {"Metric": "Time of Peak LEFT", "Value": round(peak_L_time, 2), "Unit": "seconds"},
        {"Metric": "Peak Cervical Rotation RIGHT", "Value": round(peak_R, 1), "Unit": "degrees"},
        {"Metric": "Time of Peak RIGHT", "Value": round(peak_R_time, 2), "Unit": "seconds"},
        {"Metric": "Peak Angular Velocity", "Value": round(peak_ang_vel, 1), "Unit": "degrees/second"},
        {"Metric": "Time of Peak Velocity", "Value": round(peak_vel_time, 2), "Unit": "seconds"},
    ]
    summary_df = pd.DataFrame(summary_rows)
    with open(OUTPUT_CSV, "a") as f:
        f.write("\n# Results Summary\n")
    summary_df.to_csv(OUTPUT_CSV, mode="a", index=False)
    print(f"CSV saved → {OUTPUT_CSV}")

    # -------------------- Annotated Video (continuous smooth overlay) --------------------
    cap = cv2.VideoCapture(str(video_path))
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    out = cv2.VideoWriter(str(OUTPUT_VIDEO), fourcc, fps, (w, h))

    peak_frames = {peak_L_frame, peak_R_frame}

    with PoseLandmarker.create_from_options(options) as landmarker:
        idx = 0
        while True:
            ret, frame = cap.read()
            if not ret:
                break

            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            mp_img = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
            res = landmarker.detect_for_video(mp_img, int(idx * 1000 / fps))

            if res.pose_landmarks and len(res.pose_landmarks) > 0:
                lms = res.pose_landmarks[0]
                for i, color in [(0, (0, 255, 255)),   # nose
                                 (7, (0, 200, 255)),   # L ear
                                 (8, (0, 200, 255)),   # R ear
                                 (11, (0, 255, 120)),  # L shoulder
                                 (12, (0, 255, 120))]: # R shoulder
                    try:
                        if lms[i].visibility > 0.4:
                            x = int(lms[i].x * w)
                            y = int(lms[i].y * h)
                            cv2.circle(frame, (x, y), 5, color, -1)
                    except Exception:
                        pass
                try:
                    if lms[7].visibility > 0.4 and lms[8].visibility > 0.4:
                        p1 = (int(lms[7].x * w), int(lms[7].y * h))
                        p2 = (int(lms[8].x * w), int(lms[8].y * h))
                        cv2.line(frame, p1, p2, (255, 180, 0), 2)
                except Exception:
                    pass

            # Smooth continuous overlay panel
            overlay = frame.copy()
            panel_h = 190
            cv2.rectangle(overlay, (6, 6), (470, panel_h), (20, 20, 20), -1)
            cv2.addWeighted(overlay, 0.68, frame, 0.32, 0, frame)
            cv2.rectangle(frame, (6, 6), (470, panel_h), (0, 220, 255), 2)

            t_cur = times[idx] if idx < len(times) else 0.0
            ang_v = yaw_s[idx] if idx < len(yaw_s) else 0.0
            vel_v = abs(ang_vel_s[idx]) if idx < len(ang_vel_s) else 0.0

            font = cv2.FONT_HERSHEY_SIMPLEX
            y0, dy = 32, 28
            cv2.putText(frame, "Cervical Rotation – Frontal", (14, y0),
                        font, 0.58, (0, 255, 255), 2)
            cv2.putText(frame, f"Time: {t_cur:5.2f} s", (14, y0 + dy),
                        font, 0.60, (255, 255, 255), 2)

            side = "L" if ang_v >= 0 else "R"
            cv2.putText(frame, f"Rotation: {abs(ang_v):5.1f} deg ({side})", (14, y0 + 2*dy),
                        font, 0.60, (80, 220, 255), 2)
            cv2.putText(frame, f"Ang. Velocity: {vel_v:5.1f} deg/s", (14, y0 + 3*dy),
                        font, 0.58, (0, 200, 255), 2)
            cv2.putText(frame, f"Peak L: {peak_L:4.1f} deg   Peak R: {peak_R:4.1f} deg",
                        (14, y0 + 4*dy + 4), font, 0.52, (180, 255, 180), 1)
            cv2.putText(frame, f"Peak Vel: {peak_ang_vel:5.1f} deg/s",
                        (14, y0 + 5*dy + 2), font, 0.52, (180, 255, 180), 1)

            if idx in peak_frames:
                label = "PEAK L" if idx == peak_L_frame else "PEAK R"
                cv2.putText(frame, label, (w - 140, 40), font, 0.7, (0, 0, 255), 2)
                cv2.circle(frame, (w - 30, 70), 12, (0, 0, 255), -1)

            if idx < calib_frames:
                cv2.putText(frame, "CALIBRATION", (w - 180, h - 20), font, 0.55, (0, 255, 100), 2)

            out.write(frame)
            idx += 1

    cap.release()
    out.release()
    print(f"Annotated video saved → {OUTPUT_VIDEO}\n")
    return summary_df

# ===================== RUN =====================
summary = process_cervical_rotation(VIDEO_PATH)
print(summary)

Video: 464×832 @ 59.9 fps (1453 frames)



W0000 00:00:1787501983.964698     234 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1787501983.995540     234 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


Calibration window: first 239 frames (~3.99 s)
Baseline yaw (median): -6.35°  (will be subtracted)
Smoothing yaw series...
 CERVICAL ROTATION (Frontal Smartphone + MediaPipe Pose)
 Metrics: Peak L / R Rotation Angle | Peak Angular Velocity
 Total Time                    :  24.23 s
 Calibration window            :   3.99 s

 Peak Cervical Rotation LEFT   :   74.4 °  @ 16.65 s
 Peak Cervical Rotation RIGHT  :   51.7 °  @  7.33 s
 Peak Angular Velocity         :  306.1 °/s  @ 20.12 s

 Detected Left peaks  (n=7): [8.1, 8.2, 70.8, 74.4, 71.6, 71.0, 70.8]
 Detected Right peaks (n=5): [51.5, 51.7, 50.4, 51.5, 50.0]

CSV saved → /kaggle/working/Cervical_Rotation_metrics.csv


W0000 00:00:1787502010.939208     246 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1787502010.970296     246 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


Annotated video saved → /kaggle/working/Cervical_Rotation_annotated.mp4

                         Metric   Value            Unit
0                    Total Time   24.23         seconds
1   Peak Cervical Rotation LEFT   74.40         degrees
2             Time of Peak LEFT   16.65         seconds
3  Peak Cervical Rotation RIGHT   51.70         degrees
4            Time of Peak RIGHT    7.33         seconds
5         Peak Angular Velocity  306.10  degrees/second
6         Time of Peak Velocity   20.12         seconds


In [8]:
!pip install -q mediapipe opencv-python scipy pandas

import cv2
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
import numpy as np
import pandas as pd
from scipy.signal import savgol_filter, find_peaks, medfilt
from pathlib import Path
import urllib.request
import os
import warnings
warnings.filterwarnings("ignore")

# ----------------------------- USER CONFIG -----------------------------
VIDEO_PATH = "/kaggle/input/datasets/sarangsharma/thoracic-axspa/thoracic.mp4"  # <-- change to your path
OUTPUT_DIR = Path("/kaggle/working/")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_VIDEO = OUTPUT_DIR / "Thoracic_Rotation_annotated.mp4"
OUTPUT_CSV  = OUTPUT_DIR / "Thoracic_Rotation_metrics.csv"
# -----------------------------------------------------------------------

MODEL_URL = "https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_lite/float16/1/pose_landmarker_lite.task"
MODEL_PATH = str(OUTPUT_DIR / "pose_landmarker_lite.task")
if not os.path.exists(MODEL_PATH):
    print("Downloading Pose Landmarker model...")
    urllib.request.urlretrieve(MODEL_URL, MODEL_PATH)
    print("Model downloaded.\n")

BaseOptions = python.BaseOptions
PoseLandmarker = vision.PoseLandmarker
PoseLandmarkerOptions = vision.PoseLandmarkerOptions
VisionRunningMode = vision.RunningMode

options = PoseLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=MODEL_PATH),
    running_mode=VisionRunningMode.VIDEO,
    num_poses=1,
    min_pose_detection_confidence=0.5,
    min_pose_presence_confidence=0.5,
    min_tracking_confidence=0.5
)

def get_xyz(lm, w, h):
    """Safe extraction of (x,y,z) in approximate pixel units. Never raises."""
    try:
        return np.array([lm.x * w, lm.y * h, lm.z * w], dtype=np.float64)
    except Exception:
        return np.array([np.nan, np.nan, np.nan], dtype=np.float64)

def fill_nan_1d(a):
    a = np.asarray(a, dtype=float)
    n = np.isnan(a)
    if n.any() and (~n).any():
        a = a.copy()
        a[n] = np.interp(np.flatnonzero(n), np.flatnonzero(~n), a[~n])
    return a

def smooth_series(y, fps, med_k=None, sav_win=None):
    """Median + Savitzky-Golay. Windows tuned for ~60 fps smartphone video."""
    y = np.asarray(y, dtype=float)
    if len(y) < 5:
        return y
    if med_k is None:
        med_k = max(7, int(0.12 * fps) | 1)
    if med_k % 2 == 0:
        med_k += 1
    k = min(med_k, len(y) if len(y) % 2 == 1 else len(y) - 1)
    k = max(3, k)
    y = medfilt(y, kernel_size=k)
    if sav_win is None:
        sav_win = max(11, int(0.25 * fps) | 1)
    if sav_win >= len(y):
        sav_win = len(y) - 1 if len(y) % 2 == 0 else len(y)
        sav_win = max(5, sav_win)
    if sav_win % 2 == 0:
        sav_win -= 1
    try:
        y = savgol_filter(y, sav_win, 2)
    except Exception:
        pass
    return y

def thoracic_yaw_deg(left_sh, right_sh):
    """
    Approximate thoracic (shoulder-girdle) rotation from frontal view.
    Uses the orientation of the L–R shoulder vector in the horizontal plane
    (atan2 of depth difference vs horizontal span).
    Positive = subject LEFT, Negative = subject RIGHT.
    Practical estimate; validate against goniometer / IMU for absolute values.
    """
    if (np.any(np.isnan(left_sh)) or np.any(np.isnan(right_sh))):
        return np.nan
    dx = left_sh[0] - right_sh[0]   # subject's left is normally higher image-x
    dz = left_sh[2] - right_sh[2]
    return float(np.degrees(np.arctan2(dz, dx)))

def process_thoracic_rotation(video_path):
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise FileNotFoundError(f"Cannot open video: {video_path}")

    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    print(f"Video: {w}×{h} @ {fps:.1f} fps ({total_frames} frames)\n")

    times = []
    lsh_list, rsh_list = [], []
    lhip_list, rhip_list = [], []
    nose_list = []

    with PoseLandmarker.create_from_options(options) as landmarker:
        idx = 0
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            mp_img = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
            res = landmarker.detect_for_video(mp_img, int(idx * 1000 / fps))

            t = idx / fps
            lsh = rsh = lhip = rhip = nose = np.array([np.nan, np.nan, np.nan])

            if res.pose_landmarks and len(res.pose_landmarks) > 0:
                lms = res.pose_landmarks[0]

                def vis(i):
                    try:
                        return float(lms[i].visibility)
                    except Exception:
                        return 0.0

                if vis(11) > 0.35:
                    lsh = get_xyz(lms[11], w, h)
                if vis(12) > 0.35:
                    rsh = get_xyz(lms[12], w, h)
                if vis(23) > 0.3:
                    lhip = get_xyz(lms[23], w, h)
                if vis(24) > 0.3:
                    rhip = get_xyz(lms[24], w, h)
                if vis(0) > 0.4:
                    nose = get_xyz(lms[0], w, h)

            times.append(t)
            lsh_list.append(lsh)
            rsh_list.append(rsh)
            lhip_list.append(lhip)
            rhip_list.append(rhip)
            nose_list.append(nose)
            idx += 1

    cap.release()

    times = np.asarray(times, dtype=float)
    lsh_arr = np.asarray(lsh_list, dtype=float)
    rsh_arr = np.asarray(rsh_list, dtype=float)
    lhip_arr = np.asarray(lhip_list, dtype=float)
    rhip_arr = np.asarray(rhip_list, dtype=float)
    nose_arr = np.asarray(nose_list, dtype=float)

    for arr in (lsh_arr, rsh_arr, lhip_arr, rhip_arr, nose_arr):
        for c in range(arr.shape[1]):
            arr[:, c] = fill_nan_1d(arr[:, c])

    # ---------- Calibration (first 4 seconds) ----------
    calib_frames = max(5, int(4.0 * fps))
    calib_frames = min(calib_frames, max(30, len(times) // 4))
    print(f"Calibration window: first {calib_frames} frames (~{calib_frames/fps:.2f} s)")

    yaw_list = [thoracic_yaw_deg(lsh_arr[i], rsh_arr[i]) for i in range(len(times))]
    yaw_raw = fill_nan_1d(np.asarray(yaw_list, dtype=float))
    baseline = float(np.nanmedian(yaw_raw[:calib_frames]))
    print(f"Baseline yaw (median): {baseline:.2f}° (will be subtracted)")

    yaw = yaw_raw - baseline
    print("Smoothing yaw series...")
    yaw_s = smooth_series(yaw, fps)
    yaw_s[:calib_frames] = 0.0

    # Angular velocity (°/s)
    ang_vel = np.gradient(yaw_s, times)
    ang_vel = np.clip(ang_vel, -400, 400)
    ang_vel_s = smooth_series(ang_vel, fps,
                              med_k=max(9, int(0.15 * fps) | 1),
                              sav_win=max(15, int(0.30 * fps) | 1))
    ang_vel_s[:calib_frames] = 0.0

    # ---------- Metrics ----------
    post = slice(calib_frames, None)
    post_yaw = yaw_s[post]
    post_t   = times[post]
    post_vel = ang_vel_s[post]

    peak_L = float(np.nanmax(post_yaw))
    peak_L_idx_rel = int(np.nanargmax(post_yaw))
    peak_L_time = float(post_t[peak_L_idx_rel])
    peak_L_frame = peak_L_idx_rel + calib_frames

    peak_R = float(-np.nanmin(post_yaw))
    peak_R_idx_rel = int(np.nanargmin(post_yaw))
    peak_R_time = float(post_t[peak_R_idx_rel])
    peak_R_frame = peak_R_idx_rel + calib_frames

    total_ROM = peak_L + peak_R
    peak_ang_vel = float(np.nanmax(np.abs(post_vel)))
    peak_vel_idx_rel = int(np.nanargmax(np.abs(post_vel)))
    peak_vel_time = float(post_t[peak_vel_idx_rel])

    total_time = float(times[-1]) if len(times) else 0.0

    min_dist = max(int(0.4 * fps), 8)
    peaks_L_rel, _ = find_peaks(post_yaw, height=15.0, distance=min_dist, prominence=8.0)
    peaks_R_rel, _ = find_peaks(-post_yaw, height=15.0, distance=min_dist, prominence=8.0)

    # -------------------- Results Summary --------------------
    print("=" * 70)
    print(" THORACIC ROTATION (Frontal Smartphone + MediaPipe Pose)")
    print(" Metrics: Peak L / R Rotation Angle | Total ROM | Peak Angular Velocity")
    print("=" * 70)
    print(f" Total Time              : {total_time:6.2f} s")
    print(f" Calibration window      : {calib_frames/fps:6.2f} s")
    print()
    print(f" Peak Thoracic Rotation LEFT  : {peak_L:6.1f} ° @ {peak_L_time:5.2f} s")
    print(f" Peak Thoracic Rotation RIGHT : {peak_R:6.1f} ° @ {peak_R_time:5.2f} s")
    print(f" Total ROM (L+R)              : {total_ROM:6.1f} °")
    print(f" Peak Angular Velocity        : {peak_ang_vel:6.1f} °/s @ {peak_vel_time:5.2f} s")
    print()
    print(f" Detected Left peaks  (n={len(peaks_L_rel)}): {[round(float(post_yaw[i]),1) for i in peaks_L_rel]}")
    print(f" Detected Right peaks (n={len(peaks_R_rel)}): {[round(float(-post_yaw[i]),1) for i in peaks_R_rel]}")
    print("=" * 70)
    print()

    # -------------------- CSV --------------------
    df = pd.DataFrame({
        "frame": np.arange(len(times)),
        "time_s": np.round(times, 3),
        "thoracic_rotation_deg": np.round(yaw_s, 2),   # + = Left, - = Right
        "angular_velocity_deg_s": np.round(ang_vel_s, 1)
    })

    with open(OUTPUT_CSV, "w") as f:
        f.write("# Thoracic Rotation Assessment – Frontal Smartphone View + MediaPipe Pose\n")
        f.write("# Convention: positive angle = subject LEFT rotation; negative = subject RIGHT\n")
        f.write("# Angle derived from shoulder-girdle orientation atan2(dz, dx)\n")
        f.write("# Initial ~4 s used for baseline calibration (forced to 0 after smoothing)\n")
        f.write("# Units: time = s, rotation = degrees, angular velocity = degrees/second\n")
        f.write(f"# Total Time = {total_time:.2f} s\n")
        f.write(f"# Peak Left = {peak_L:.1f} deg @ {peak_L_time:.2f} s\n")
        f.write(f"# Peak Right = {peak_R:.1f} deg @ {peak_R_time:.2f} s\n")
        f.write(f"# Total ROM = {total_ROM:.1f} deg\n")
        f.write(f"# Peak Angular Velocity = {peak_ang_vel:.1f} deg/s\n\n")

    df.to_csv(OUTPUT_CSV, mode="a", index=False)

    summary_rows = [
        {"Metric": "Total Time", "Value": round(total_time, 2), "Unit": "seconds"},
        {"Metric": "Peak Thoracic Rotation LEFT", "Value": round(peak_L, 1), "Unit": "degrees"},
        {"Metric": "Time of Peak LEFT", "Value": round(peak_L_time, 2), "Unit": "seconds"},
        {"Metric": "Peak Thoracic Rotation RIGHT", "Value": round(peak_R, 1), "Unit": "degrees"},
        {"Metric": "Time of Peak RIGHT", "Value": round(peak_R_time, 2), "Unit": "seconds"},
        {"Metric": "Total ROM (L+R)", "Value": round(total_ROM, 1), "Unit": "degrees"},
        {"Metric": "Peak Angular Velocity", "Value": round(peak_ang_vel, 1), "Unit": "degrees/second"},
        {"Metric": "Time of Peak Velocity", "Value": round(peak_vel_time, 2), "Unit": "seconds"},
    ]
    summary_df = pd.DataFrame(summary_rows)
    with open(OUTPUT_CSV, "a") as f:
        f.write("\n# Results Summary\n")
    summary_df.to_csv(OUTPUT_CSV, mode="a", index=False)
    print(f"CSV saved → {OUTPUT_CSV}")

    # -------------------- Annotated Video (continuous smooth overlay) --------------------
    cap = cv2.VideoCapture(str(video_path))
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    out = cv2.VideoWriter(str(OUTPUT_VIDEO), fourcc, fps, (w, h))

    peak_frames = {peak_L_frame, peak_R_frame}

    with PoseLandmarker.create_from_options(options) as landmarker:
        idx = 0
        while True:
            ret, frame = cap.read()
            if not ret:
                break

            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            mp_img = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
            res = landmarker.detect_for_video(mp_img, int(idx * 1000 / fps))

            if res.pose_landmarks and len(res.pose_landmarks) > 0:
                lms = res.pose_landmarks[0]
                # key points: nose, shoulders, hips, elbows (crossed-arms context)
                for i, color in [
                    (0, (0, 255, 255)),   # nose
                    (11, (0, 255, 120)),  # L shoulder
                    (12, (0, 255, 120)),  # R shoulder
                    (23, (255, 180, 0)),  # L hip
                    (24, (255, 180, 0)),  # R hip
                    (13, (200, 100, 255)),# L elbow
                    (14, (200, 100, 255)),# R elbow
                ]:
                    try:
                        if lms[i].visibility > 0.35:
                            x = int(lms[i].x * w)
                            y = int(lms[i].y * h)
                            cv2.circle(frame, (x, y), 5, color, -1)
                    except Exception:
                        pass

                # shoulder line
                try:
                    if lms[11].visibility > 0.35 and lms[12].visibility > 0.35:
                        p1 = (int(lms[11].x * w), int(lms[11].y * h))
                        p2 = (int(lms[12].x * w), int(lms[12].y * h))
                        cv2.line(frame, p1, p2, (0, 255, 120), 2)
                except Exception:
                    pass
                # hip line
                try:
                    if lms[23].visibility > 0.3 and lms[24].visibility > 0.3:
                        p1 = (int(lms[23].x * w), int(lms[23].y * h))
                        p2 = (int(lms[24].x * w), int(lms[24].y * h))
                        cv2.line(frame, p1, p2, (255, 180, 0), 2)
                except Exception:
                    pass

            # Smooth continuous overlay panel
            overlay = frame.copy()
            panel_h = 210
            cv2.rectangle(overlay, (6, 6), (490, panel_h), (20, 20, 20), -1)
            cv2.addWeighted(overlay, 0.68, frame, 0.32, 0, frame)
            cv2.rectangle(frame, (6, 6), (490, panel_h), (0, 220, 255), 2)

            t_cur = times[idx] if idx < len(times) else 0.0
            ang_v = yaw_s[idx] if idx < len(yaw_s) else 0.0
            vel_v = abs(ang_vel_s[idx]) if idx < len(ang_vel_s) else 0.0

            font = cv2.FONT_HERSHEY_SIMPLEX
            y0, dy = 32, 28
            cv2.putText(frame, "Thoracic Rotation – Frontal", (14, y0),
                        font, 0.58, (0, 255, 255), 2)
            cv2.putText(frame, f"Time: {t_cur:5.2f} s", (14, y0 + dy),
                        font, 0.60, (255, 255, 255), 2)
            side = "L" if ang_v >= 0 else "R"
            cv2.putText(frame, f"Rotation: {abs(ang_v):5.1f} deg ({side})", (14, y0 + 2*dy),
                        font, 0.60, (80, 220, 255), 2)
            cv2.putText(frame, f"Ang. Velocity: {vel_v:5.1f} deg/s", (14, y0 + 3*dy),
                        font, 0.58, (0, 200, 255), 2)
            cv2.putText(frame, f"Peak L: {peak_L:4.1f}  Peak R: {peak_R:4.1f}  ROM: {total_ROM:4.1f}",
                        (14, y0 + 4*dy + 4), font, 0.50, (180, 255, 180), 1)
            cv2.putText(frame, f"Peak Vel: {peak_ang_vel:5.1f} deg/s",
                        (14, y0 + 5*dy + 2), font, 0.52, (180, 255, 180), 1)

            if idx in peak_frames:
                label = "PEAK L" if idx == peak_L_frame else "PEAK R"
                cv2.putText(frame, label, (w - 140, 40), font, 0.7, (0, 0, 255), 2)
                cv2.circle(frame, (w - 30, 70), 12, (0, 0, 255), -1)

            if idx < calib_frames:
                cv2.putText(frame, "CALIBRATION", (w - 180, h - 20), font, 0.55, (0, 255, 100), 2)

            out.write(frame)
            idx += 1

    cap.release()
    out.release()
    print(f"Annotated video saved → {OUTPUT_VIDEO}\n")
    return summary_df

# ===================== RUN =====================
summary = process_thoracic_rotation(VIDEO_PATH)
print(summary)

Video: 464×832 @ 59.9 fps (1529 frames)



W0000 00:00:1787502048.722943     264 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1787502048.753554     264 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


Calibration window: first 239 frames (~3.99 s)
Baseline yaw (median): -8.96° (will be subtracted)
Smoothing yaw series...
 THORACIC ROTATION (Frontal Smartphone + MediaPipe Pose)
 Metrics: Peak L / R Rotation Angle | Total ROM | Peak Angular Velocity
 Total Time              :  25.50 s
 Calibration window      :   3.99 s

 Peak Thoracic Rotation LEFT  :   78.6 ° @ 16.47 s
 Peak Thoracic Rotation RIGHT :   65.4 ° @ 14.87 s
 Total ROM (L+R)              :  144.1 °
 Peak Angular Velocity        :  318.1 °/s @  6.91 s

 Detected Left peaks  (n=9): [20.4, 25.8, 27.5, 17.4, 78.6, 68.2, 68.8, 68.3, 73.0]
 Detected Right peaks (n=8): [55.0, 56.4, 29.2, 64.6, 63.1, 65.4, 31.0, 26.0]

CSV saved → /kaggle/working/Thoracic_Rotation_metrics.csv


W0000 00:00:1787502076.617109     279 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1787502076.642916     278 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


Annotated video saved → /kaggle/working/Thoracic_Rotation_annotated.mp4

                         Metric   Value            Unit
0                    Total Time   25.50         seconds
1   Peak Thoracic Rotation LEFT   78.60         degrees
2             Time of Peak LEFT   16.47         seconds
3  Peak Thoracic Rotation RIGHT   65.40         degrees
4            Time of Peak RIGHT   14.87         seconds
5               Total ROM (L+R)  144.10         degrees
6         Peak Angular Velocity  318.10  degrees/second
7         Time of Peak Velocity    6.91         seconds


In [9]:
# Getting Up From Floor Assessment (2 repetitions)
# Smartphone frontal view + MediaPipe Pose Landmarker
# Metrics: time (mean of 2 trials), lumbar (torso elevation) / hip angles (°)
# Continuous smooth overlay on annotated video + CSV + summary.
#
# In a Kaggle / Colab notebook cell, first run:
# !pip install -q mediapipe opencv-python scipy pandas

import cv2
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
import numpy as np
import pandas as pd
from scipy.signal import savgol_filter, find_peaks, medfilt
from pathlib import Path
import urllib.request
import os
import warnings
warnings.filterwarnings("ignore")

# ----------------------------- USER CONFIG -----------------------------
VIDEO_PATH = "/kaggle/input/datasets/sarangsharma/lying-axspa/lying to stand.mp4"   # <-- change
CALIB_PATH = "/kaggle/input/datasets/sarangsharma/lying-axspa/lyingcalib.mp4"       # <-- change
# Local test example:
# VIDEO_PATH = "/home/workdir/attachments/lying to stand.mp4"
# CALIB_PATH = "/home/workdir/attachments/lyingcalib.mp4"

OUTPUT_DIR = Path("/kaggle/working/")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_VIDEO = OUTPUT_DIR / "GettingUp_From_Floor_annotated.mp4"
OUTPUT_CSV   = OUTPUT_DIR / "GettingUp_From_Floor_metrics.csv"
# -----------------------------------------------------------------------

MODEL_URL = "https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_lite/float16/1/pose_landmarker_lite.task"
MODEL_PATH = str(OUTPUT_DIR / "pose_landmarker_lite.task")
if not os.path.exists(MODEL_PATH):
    print("Downloading Pose Landmarker model...")
    urllib.request.urlretrieve(MODEL_URL, MODEL_PATH)
    print("Model downloaded.\n")

BaseOptions = python.BaseOptions
PoseLandmarker = vision.PoseLandmarker
PoseLandmarkerOptions = vision.PoseLandmarkerOptions
VisionRunningMode = vision.RunningMode

options = PoseLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=MODEL_PATH),
    running_mode=VisionRunningMode.VIDEO,
    num_poses=1,
    min_pose_detection_confidence=0.45,
    min_pose_presence_confidence=0.45,
    min_tracking_confidence=0.45
)

def get_px(lm, w, h):
    return np.array([lm.x * w, lm.y * h], dtype=np.float64)

def fill_nan_1d(a):
    a = np.asarray(a, dtype=float)
    n = np.isnan(a)
    if n.any() and (~n).any():
        a = a.copy()
        a[n] = np.interp(np.flatnonzero(n), np.flatnonzero(~n), a[~n])
    return a

def smooth_series(y, fps, med_k=None, sav_win=None):
    """Median + Savitzky-Golay. Windows tuned for ~60 fps smartphone video."""
    y = np.asarray(y, dtype=float)
    if len(y) < 5:
        return y
    if med_k is None:
        med_k = max(7, int(0.10 * fps) | 1)
    if med_k % 2 == 0:
        med_k += 1
    k = min(med_k, len(y) if len(y) % 2 == 1 else len(y) - 1)
    k = max(3, k)
    y = medfilt(y, kernel_size=k)
    if sav_win is None:
        sav_win = max(11, int(0.22 * fps) | 1)
    if sav_win >= len(y):
        sav_win = len(y) - 1 if len(y) % 2 == 0 else len(y)
        sav_win = max(5, sav_win)
    if sav_win % 2 == 0:
        sav_win -= 1
    try:
        y = savgol_filter(y, sav_win, 2)
    except Exception:
        pass
    return y

def torso_elev_deg(sh, hip):
    """
    Torso elevation (lumbar proxy) from horizontal, degrees.
    0° ≈ lying horizontal, ~90° ≈ upright standing.
    Robust to frontal or side projection.
    """
    if np.any(np.isnan(sh)) or np.any(np.isnan(hip)):
        return np.nan
    dx = sh[0] - hip[0]
    dy = sh[1] - hip[1]          # image +y downward
    length = np.hypot(dx, dy)
    if length < 8.0:
        return np.nan
    vert = np.clip(-dy / length, -1.0, 1.0)
    return float(np.degrees(np.arcsin(vert)))

def hip_angle_deg(sh, hip, knee):
    """
    Hip joint angle (°) between torso (hip→shoulder) and thigh (hip→knee).
    ~180° extended; decreases with flexion.
    """
    if (np.any(np.isnan(sh)) or np.any(np.isnan(hip)) or np.any(np.isnan(knee))):
        return np.nan
    v1 = sh - hip
    v2 = knee - hip
    n1 = np.linalg.norm(v1)
    n2 = np.linalg.norm(v2)
    if n1 < 8.0 or n2 < 8.0:
        return np.nan
    cosang = np.dot(v1, v2) / (n1 * n2)
    return float(np.degrees(np.arccos(np.clip(cosang, -1.0, 1.0))))

def extract_pose_series(video_path, label="video"):
    """Run MediaPipe VIDEO mode → times + key landmark series + angles."""
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise FileNotFoundError(f"Cannot open {label}: {video_path}")
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    print(f"{label}: {w}×{h} @ {fps:.1f} fps ({total} frames)")

    times, elev_list, hipa_list, hipy_list = [], [], [], []
    sh_list, hip_list, kn_list = [], [], []

    with PoseLandmarker.create_from_options(options) as landmarker:
        idx = 0
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            mp_img = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
            res = landmarker.detect_for_video(mp_img, int(idx * 1000 / fps))
            t = idx / fps

            sh = hip = kn = np.array([np.nan, np.nan])
            elev = hipa = hy = np.nan

            if res.pose_landmarks and len(res.pose_landmarks) > 0:
                lms = res.pose_landmarks[0]
                def vis(i):
                    try:
                        return float(lms[i].visibility)
                    except Exception:
                        return 0.0

                # Prefer mid-points; fall back to single visible side
                if vis(11) > 0.35 and vis(12) > 0.35:
                    sh = 0.5 * (get_px(lms[11], w, h) + get_px(lms[12], w, h))
                elif vis(11) > 0.35:
                    sh = get_px(lms[11], w, h)
                elif vis(12) > 0.35:
                    sh = get_px(lms[12], w, h)

                if vis(23) > 0.35 and vis(24) > 0.35:
                    hip = 0.5 * (get_px(lms[23], w, h) + get_px(lms[24], w, h))
                elif vis(23) > 0.35:
                    hip = get_px(lms[23], w, h)
                elif vis(24) > 0.35:
                    hip = get_px(lms[24], w, h)

                if vis(25) > 0.25 and vis(26) > 0.25:
                    kn = 0.5 * (get_px(lms[25], w, h) + get_px(lms[26], w, h))
                elif vis(25) > 0.25:
                    kn = get_px(lms[25], w, h)
                elif vis(26) > 0.25:
                    kn = get_px(lms[26], w, h)

                elev = torso_elev_deg(sh, hip)
                hipa = hip_angle_deg(sh, hip, kn)
                if not np.any(np.isnan(hip)):
                    hy = float(hip[1] / h)

            times.append(t)
            elev_list.append(elev)
            hipa_list.append(hipa)
            hipy_list.append(hy)
            sh_list.append(sh)
            hip_list.append(hip)
            kn_list.append(kn)
            idx += 1

    cap.release()
    times = np.asarray(times, dtype=float)
    elev = fill_nan_1d(np.asarray(elev_list, dtype=float))
    hipa = fill_nan_1d(np.asarray(hipa_list, dtype=float))
    hipy = fill_nan_1d(np.asarray(hipy_list, dtype=float))
    elev = np.clip(elev, -20.0, 95.0)

    return {
        "fps": fps, "w": w, "h": h,
        "times": times,
        "elev": elev, "hipa": hipa, "hipy": hipy,
        "sh": np.asarray(sh_list, dtype=float),
        "hip": np.asarray(hip_list, dtype=float),
        "kn": np.asarray(kn_list, dtype=float),
    }

def detect_two_trials(times, elev_s, hipy_s, fps):
    """
    Detect two floor-to-stand trials.
    Robust to end-of-video standing plateau (no descent after 2nd rise).
    """
    n = len(elev_s)
    min_dist = max(int(0.9 * fps), 35)
    peaks, _ = find_peaks(elev_s, height=58.0, distance=min_dist, prominence=10.0)

    # Sustained high plateaus (captures final standing that never comes back down)
    high = elev_s > 65.0
    plateau_peaks = []
    i = 0
    while i < n:
        if high[i]:
            j = i
            while j < n and high[j]:
                j += 1
            if (j - i) >= int(0.55 * fps):
                local = i + int(np.argmax(elev_s[i:j]))
                plateau_peaks.append(local)
            i = j
        else:
            i += 1

    all_cand_idx = sorted(set(list(peaks) + plateau_peaks))

    if len(all_cand_idx) < 2:
        inv_h = -hipy_s
        p2, _ = find_peaks(inv_h, height=-0.53, distance=min_dist, prominence=0.03)
        all_cand_idx = sorted(set(all_cand_idx) | set(p2.tolist()))

    candidates = []
    for p in all_cand_idx:
        if elev_s[p] < 48:
            continue
        start = int(p)
        search_lim = max(0, p - int(5.0 * fps))
        for j in range(p, search_lim, -1):
            if elev_s[j] < 22.0:
                start = j
                break
            if j < p - 6 and hipy_s[j] > hipy_s[min(j + 3, n-1)] + 0.02:
                start = j
                break
        for j in range(start, max(0, start - int(0.3 * fps)), -1):
            if elev_s[j] < elev_s[start] - 2.0:
                start = j
            else:
                break

        end = int(p)
        for j in range(p, min(n - 1, p + int(2.0 * fps))):
            if elev_s[j] > 52.0:
                end = j
            else:
                break
        if elev_s[-1] > 70 and end > n - int(0.4 * fps):
            end = n - 1

        dur = float(times[end] - times[start])
        if 0.65 < dur < 10.0:
            candidates.append({
                "start_idx": int(start),
                "end_idx": int(end),
                "peak_idx": int(p),
                "start_t": float(times[start]),
                "end_t": float(times[end]),
                "peak_t": float(times[p]),
                "peak_elev": float(elev_s[p]),
                "duration": dur,
            })

    candidates = sorted(candidates, key=lambda c: (-c["peak_elev"], c["start_t"]))
    trials = []
    used = []
    for c in candidates:
        if any(abs(c["start_t"] - u) < 1.6 for u in used):
            continue
        trials.append(c)
        used.append(c["start_t"])
        if len(trials) >= 2:
            break

    return sorted(trials, key=lambda c: c["start_t"])

def process_getting_up(video_path, calib_path=None):
    baseline_elev = 0.0
    if calib_path and os.path.exists(str(calib_path)):
        print("Processing calibration (static lying)...")
        cal = extract_pose_series(calib_path, label="Calib")
        baseline_elev = float(np.nanmedian(cal["elev"]))
        print(f"  Calib baseline: torso elev = {baseline_elev:.1f}°")
        print(f"  Stability (elev std) = {np.nanstd(cal['elev']):.2f}°  → good\n")
    else:
        print("No calibration video – using absolute angles.\n")

    print("Processing Getting-Up-From-Floor video...")
    data = extract_pose_series(video_path, label="Main")
    fps, w, h = data["fps"], data["w"], data["h"]
    times = data["times"]
    elev_raw = data["elev"] - baseline_elev
    hipa_raw = data["hipa"]
    hipy = data["hipy"]

    print("Smoothing series...")
    elev_s = smooth_series(elev_raw, fps)
    hipa_s = smooth_series(hipa_raw, fps)
    hipy_s = smooth_series(hipy, fps, med_k=max(5, int(0.08*fps)|1))
    early = max(5, int(0.4 * fps))
    elev_s[:early] = np.minimum(elev_s[:early], 5.0)

    trials = detect_two_trials(times, elev_s, hipy_s, fps)
    print(f"Detected {len(trials)} trial(s)")

    trial_times = []
    for i, tr in enumerate(trials):
        trial_times.append(tr["duration"])
        print(f"  Trial {i+1}: {tr['start_t']:.2f}s → {tr['end_t']:.2f}s  "
              f"(dur={tr['duration']:.2f}s, peak elev={tr['peak_elev']:.1f}° @ {tr['peak_t']:.2f}s)")

    mean_time = float(np.mean(trial_times)) if trial_times else float("nan")
    total_time = float(times[-1]) if len(times) else 0.0

    peak_elev = float(np.nanmax(elev_s))
    peak_elev_t = float(times[np.nanargmax(elev_s)])
    min_hipa = float(np.nanmin(hipa_s))
    min_hipa_t = float(times[np.nanargmin(hipa_s)])
    max_hipa = float(np.nanmax(hipa_s))

    print("=" * 72)
    print(" GETTING UP FROM FLOOR – 2 Repetitions")
    print(" Smartphone Frontal View + MediaPipe Pose")
    print(" Metrics: Time (mean of 2 trials) | Lumbar (torso elev) / Hip angles")
    print("=" * 72)
    print(f" Total video duration          : {total_time:6.2f} s")
    print(f" Number of trials detected     : {len(trials)}")
    if len(trials) >= 1:
        print(f" Trial 1 duration              : {trials[0]['duration']:6.2f} s")
    if len(trials) >= 2:
        print(f" Trial 2 duration              : {trials[1]['duration']:6.2f} s")
    print(f" MEAN TIME (2 trials)          : {mean_time:6.2f} s")
    print()
    print(f" Peak torso elevation (lumbar) : {peak_elev:6.1f} ° @ {peak_elev_t:5.2f} s")
    print(f" Max hip extension             : {max_hipa:6.1f} °")
    print(f" Max hip flexion (min angle)   : {min_hipa:6.1f} ° @ {min_hipa_t:5.2f} s")
    print("=" * 72)
    print()

    # CSV
    df = pd.DataFrame({
        "frame": np.arange(len(times)),
        "time_s": np.round(times, 3),
        "torso_elev_deg": np.round(elev_s, 2),
        "hip_angle_deg": np.round(hipa_s, 1),
        "hip_y_norm": np.round(hipy_s, 4),
    })

    with open(OUTPUT_CSV, "w") as f:
        f.write("# Getting Up From Floor Assessment – 2 Repetitions\n")
        f.write("# Frontal smartphone view + MediaPipe Pose Landmarker\n")
        f.write("# Torso elev: 0° ≈ horizontal lying, ~90° ≈ upright (lumbar proxy)\n")
        f.write("# Hip angle: ~180° extended, lower = more flexed\n")
        f.write(f"# Mean Time (2 trials) = {mean_time:.2f} s\n")
        if len(trials) >= 1:
            f.write(f"# Trial 1 = {trials[0]['duration']:.2f} s\n")
        if len(trials) >= 2:
            f.write(f"# Trial 2 = {trials[1]['duration']:.2f} s\n")
        f.write(f"# Peak torso elev = {peak_elev:.1f} deg\n")
        f.write(f"# Max hip flexion (min angle) = {min_hipa:.1f} deg\n\n")

    df.to_csv(OUTPUT_CSV, mode="a", index=False)

    summary_rows = [
        {"Metric": "Mean Time (2 trials)", "Value": round(mean_time, 2), "Unit": "seconds"},
    ]
    if len(trials) >= 1:
        summary_rows.append({"Metric": "Trial 1 Duration", "Value": round(trials[0]["duration"], 2), "Unit": "seconds"})
    if len(trials) >= 2:
        summary_rows.append({"Metric": "Trial 2 Duration", "Value": round(trials[1]["duration"], 2), "Unit": "seconds"})
    summary_rows.extend([
        {"Metric": "Peak Torso Elevation (lumbar)", "Value": round(peak_elev, 1), "Unit": "degrees"},
        {"Metric": "Time of Peak Elevation", "Value": round(peak_elev_t, 2), "Unit": "seconds"},
        {"Metric": "Max Hip Extension", "Value": round(max_hipa, 1), "Unit": "degrees"},
        {"Metric": "Max Hip Flexion (min angle)", "Value": round(min_hipa, 1), "Unit": "degrees"},
        {"Metric": "Time of Max Hip Flexion", "Value": round(min_hipa_t, 2), "Unit": "seconds"},
        {"Metric": "Total Video Duration", "Value": round(total_time, 2), "Unit": "seconds"},
    ])
    summary_df = pd.DataFrame(summary_rows)
    with open(OUTPUT_CSV, "a") as f:
        f.write("\n# Results Summary\n")
    summary_df.to_csv(OUTPUT_CSV, mode="a", index=False)
    print(f"CSV saved → {OUTPUT_CSV}")

    # Annotated video – continuous smooth overlay
    print("Writing annotated video with continuous overlay...")
    cap = cv2.VideoCapture(str(video_path))
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    out = cv2.VideoWriter(str(OUTPUT_VIDEO), fourcc, fps, (w, h))

    trial_ranges = [(tr["start_idx"], tr["end_idx"], tr["duration"]) for tr in trials]

    with PoseLandmarker.create_from_options(options) as landmarker:
        idx = 0
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            mp_img = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
            res = landmarker.detect_for_video(mp_img, int(idx * 1000 / fps))

            if res.pose_landmarks and len(res.pose_landmarks) > 0:
                lms = res.pose_landmarks[0]
                def vis(i):
                    try:
                        return float(lms[i].visibility)
                    except Exception:
                        return 0.0
                pts = {}
                for i, col in [(0, (0, 255, 255)),
                               (11, (0, 255, 120)), (12, (0, 255, 120)),
                               (23, (0, 200, 255)), (24, (0, 200, 255)),
                               (25, (255, 180, 0)), (26, (255, 180, 0)),
                               (27, (200, 100, 255)), (28, (200, 100, 255))]:
                    if vis(i) > 0.3:
                        x = int(lms[i].x * w)
                        y = int(lms[i].y * h)
                        cv2.circle(frame, (x, y), 5, col, -1)
                        pts[i] = (x, y)
                for a, b in [(11, 12), (11, 23), (12, 24), (23, 24),
                             (23, 25), (24, 26), (25, 27), (26, 28)]:
                    if a in pts and b in pts:
                        cv2.line(frame, pts[a], pts[b], (0, 220, 255), 2)

            # Continuous overlay panel
            overlay = frame.copy()
            panel_h = 210
            cv2.rectangle(overlay, (6, 6), (480, panel_h), (15, 15, 15), -1)
            cv2.addWeighted(overlay, 0.72, frame, 0.28, 0, frame)
            cv2.rectangle(frame, (6, 6), (480, panel_h), (0, 220, 255), 2)

            t_cur = times[idx] if idx < len(times) else 0.0
            elev_v = elev_s[idx] if idx < len(elev_s) else 0.0
            hipa_v = hipa_s[idx] if idx < len(hipa_s) else 180.0

            font = cv2.FONT_HERSHEY_SIMPLEX
            y0, dy = 30, 26
            cv2.putText(frame, "Getting Up From Floor (2 reps)", (14, y0),
                        font, 0.55, (0, 255, 255), 2)
            cv2.putText(frame, f"Time: {t_cur:5.2f} s", (14, y0 + dy),
                        font, 0.58, (255, 255, 255), 2)
            cv2.putText(frame, f"Torso elev (lumbar): {elev_v:5.1f} deg", (14, y0 + 2*dy),
                        font, 0.56, (80, 220, 255), 2)
            cv2.putText(frame, f"Hip angle: {hipa_v:5.1f} deg", (14, y0 + 3*dy),
                        font, 0.56, (0, 200, 255), 2)
            cv2.putText(frame, f"Mean time (2 trials): {mean_time:5.2f} s",
                        (14, y0 + 4*dy + 4), font, 0.52, (180, 255, 180), 1)
            if len(trials) >= 2:
                cv2.putText(frame, f"T1: {trials[0]['duration']:.2f}s   T2: {trials[1]['duration']:.2f}s",
                            (14, y0 + 5*dy + 2), font, 0.50, (180, 255, 180), 1)

            for ti, (sidx, eidx, dur) in enumerate(trial_ranges):
                if sidx <= idx <= eidx:
                    cv2.putText(frame, f"TRIAL {ti+1}  ({dur:.2f}s)", (w - 210, 36),
                                font, 0.65, (0, 80, 255), 2)
                    prog = (idx - sidx) / max(1, eidx - sidx)
                    bar_w = int(180 * prog)
                    cv2.rectangle(frame, (w - 200, 48), (w - 20, 62), (40, 40, 40), -1)
                    cv2.rectangle(frame, (w - 200, 48), (w - 200 + bar_w, 62), (0, 180, 255), -1)
                    break

            if abs(t_cur - peak_elev_t) < 0.08:
                cv2.putText(frame, "PEAK ELEV", (w - 160, 90), font, 0.6, (0, 0, 255), 2)

            out.write(frame)
            idx += 1

    cap.release()
    out.release()
    print(f"Annotated video saved → {OUTPUT_VIDEO}\n")
    return summary_df, trials

# ===================== RUN =====================
summary, trials = process_getting_up(VIDEO_PATH, CALIB_PATH)
print(summary)

Processing calibration (static lying)...
Calib: 464×832 @ 59.9 fps (349 frames)


W0000 00:00:1787502110.432263     290 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1787502110.457161     292 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


  Calib baseline: torso elev = -1.1°
  Stability (elev std) = 0.13°  → good

Processing Getting-Up-From-Floor video...
Main: 464×832 @ 59.9 fps (564 frames)


W0000 00:00:1787502117.232040     303 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1787502117.259379     303 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


Smoothing series...
Detected 2 trial(s)
  Trial 1: 3.20s → 4.67s  (dur=1.47s, peak elev=97.3° @ 4.61s)
  Trial 2: 7.54s → 9.39s  (dur=1.85s, peak elev=90.8° @ 8.48s)
 GETTING UP FROM FLOOR – 2 Repetitions
 Smartphone Frontal View + MediaPipe Pose
 Metrics: Time (mean of 2 trials) | Lumbar (torso elev) / Hip angles
 Total video duration          :   9.39 s
 Number of trials detected     : 2
 Trial 1 duration              :   1.47 s
 Trial 2 duration              :   1.85 s
 MEAN TIME (2 trials)          :   1.66 s

 Peak torso elevation (lumbar) :   97.3 ° @  4.61 s
 Max hip extension             :  179.9 °
 Max hip flexion (min angle)   :   40.5 ° @  2.95 s

CSV saved → /kaggle/working/GettingUp_From_Floor_metrics.csv
Writing annotated video with continuous overlay...


W0000 00:00:1787502128.564030     317 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1787502128.588571     318 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


Annotated video saved → /kaggle/working/GettingUp_From_Floor_annotated.mp4

                          Metric   Value     Unit
0           Mean Time (2 trials)    1.66  seconds
1               Trial 1 Duration    1.47  seconds
2               Trial 2 Duration    1.85  seconds
3  Peak Torso Elevation (lumbar)   97.30  degrees
4         Time of Peak Elevation    4.61  seconds
5              Max Hip Extension  179.90  degrees
6    Max Hip Flexion (min angle)   40.50  degrees
7        Time of Max Hip Flexion    2.95  seconds
8           Total Video Duration    9.39  seconds


In [10]:
!pip install -q mediapipe opencv-python scipy pandas

import cv2
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
import numpy as np
import pandas as pd
from scipy.signal import savgol_filter, find_peaks, medfilt
from pathlib import Path
import urllib.request
import os
import warnings
warnings.filterwarnings("ignore")

# ----------------------------- USER CONFIG -----------------------------
VIDEO_PATH = "/kaggle/input/datasets/sarangsharma/chair-axspa/chairstand.mp4"  # <-- change to your path
OUTPUT_DIR = Path("/kaggle/working/")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_VIDEO = OUTPUT_DIR / "30s_ChairStand_annotated.mp4"
OUTPUT_CSV  = OUTPUT_DIR / "30s_ChairStand_metrics.csv"
# -----------------------------------------------------------------------

MODEL_URL = "https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_lite/float16/1/pose_landmarker_lite.task"
MODEL_PATH = str(OUTPUT_DIR / "pose_landmarker_lite.task")
if not os.path.exists(MODEL_PATH):
    print("Downloading Pose Landmarker model...")
    urllib.request.urlretrieve(MODEL_URL, MODEL_PATH)
    print("Model downloaded.\n")

BaseOptions = python.BaseOptions
PoseLandmarker = vision.PoseLandmarker
PoseLandmarkerOptions = vision.PoseLandmarkerOptions
VisionRunningMode = vision.RunningMode

options = PoseLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=MODEL_PATH),
    running_mode=VisionRunningMode.VIDEO,
    num_poses=1,
    min_pose_detection_confidence=0.5,
    min_pose_presence_confidence=0.5,
    min_tracking_confidence=0.5
)

def get_px(lm, w, h):
    return np.array([lm.x * w, lm.y * h], dtype=np.float64)

def fill_nan_1d(a):
    a = np.asarray(a, dtype=float)
    n = np.isnan(a)
    if n.any() and (~n).any():
        a = a.copy()
        a[n] = np.interp(np.flatnonzero(n), np.flatnonzero(~n), a[~n])
    return a

def smooth_series(y, fps, med_k=None, sav_win=None):
    """Median + Savitzky-Golay. Windows tuned for ~60 fps smartphone video."""
    y = np.asarray(y, dtype=float)
    if len(y) < 5:
        return y
    if med_k is None:
        med_k = max(7, int(0.10 * fps) | 1)
    if med_k % 2 == 0:
        med_k += 1
    k = min(med_k, len(y) if len(y) % 2 == 1 else len(y) - 1)
    k = max(3, k)
    y = medfilt(y, kernel_size=k)
    if sav_win is None:
        sav_win = max(11, int(0.22 * fps) | 1)
    if sav_win >= len(y):
        sav_win = len(y) - 1 if len(y) % 2 == 0 else len(y)
        sav_win = max(5, sav_win)
    if sav_win % 2 == 0:
        sav_win -= 1
    try:
        y = savgol_filter(y, sav_win, 2)
    except Exception:
        pass
    return y

def process_30s_chair_stand(video_path):
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise FileNotFoundError(f"Cannot open video: {video_path}")

    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    print(f"Video: {w}×{h} @ {fps:.1f} fps ({total_frames} frames)\n")

    times, hip_y_list, sh_y_list, knee_y_list, trunk_len_list = [], [], [], [], []

    with PoseLandmarker.create_from_options(options) as landmarker:
        idx = 0
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            mp_img = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
            res = landmarker.detect_for_video(mp_img, int(idx * 1000 / fps))
            t = idx / fps

            mid_hip_y = mid_sh_y = mid_knee_y = np.nan
            if res.pose_landmarks and len(res.pose_landmarks) > 0:
                lms = res.pose_landmarks[0]
                def vis(i):
                    try:
                        return float(lms[i].visibility)
                    except Exception:
                        return 0.0
                try:
                    if vis(23) > 0.40 and vis(24) > 0.40:
                        mid_hip_y = 0.5 * (lms[23].y + lms[24].y) * h
                    if vis(11) > 0.40 and vis(12) > 0.40:
                        mid_sh_y = 0.5 * (lms[11].y + lms[12].y) * h
                    if vis(25) > 0.30 and vis(26) > 0.30:
                        mid_knee_y = 0.5 * (lms[25].y + lms[26].y) * h
                except Exception:
                    pass

            times.append(t)
            hip_y_list.append(mid_hip_y)
            sh_y_list.append(mid_sh_y)
            knee_y_list.append(mid_knee_y)
            if not (np.isnan(mid_hip_y) or np.isnan(mid_sh_y)):
                trunk_len_list.append(abs(mid_sh_y - mid_hip_y))
            else:
                trunk_len_list.append(np.nan)
            idx += 1

    cap.release()
    times = np.asarray(times, dtype=float)
    hip_y = fill_nan_1d(np.asarray(hip_y_list, dtype=float))
    sh_y  = fill_nan_1d(np.asarray(sh_y_list, dtype=float))
    knee_y = fill_nan_1d(np.asarray(knee_y_list, dtype=float))
    trunk_len = fill_nan_1d(np.asarray(trunk_len_list, dtype=float))

    # ---------- Calibration (first 2 seconds – seated baseline) ----------
    calib_frames = max(5, int(2.0 * fps))
    calib_frames = min(calib_frames, max(20, len(times) // 5))
    print(f"Calibration window: first {calib_frames} frames (~{calib_frames/fps:.2f} s)")

    baseline_hip = float(np.nanmedian(hip_y[:calib_frames]))
    print(f"Baseline hip_y (seated): {baseline_hip:.1f} px")

    # Standing-height signal (positive = risen above seated baseline)
    stand_h = baseline_hip - hip_y
    stand_h_s = smooth_series(stand_h, fps)
    stand_h_s[:calib_frames] = 0.0

    # Trunk flexion approximation (°): reduction of projected shoulder–hip length
    # (0° ≈ upright; increases with forward lean during the STS transition)
    L0 = float(np.nanmedian(trunk_len[:calib_frames]))
    L0 = max(L0, 1.0)
    print(f"Baseline trunk projected length: {L0:.1f} px")
    trunk_flex = np.degrees(np.arccos(np.clip(trunk_len / L0, 0.0, 1.0)))
    trunk_flex = fill_nan_1d(trunk_flex)
    trunk_flex_s = smooth_series(trunk_flex, fps)
    trunk_flex_s[:calib_frames] = 0.0

    # Angular velocity of trunk flexion
    ang_vel = np.gradient(trunk_flex_s, times)
    ang_vel = np.clip(ang_vel, -200, 200)
    ang_vel_s = smooth_series(ang_vel, fps,
                              med_k=max(9, int(0.12 * fps) | 1),
                              sav_win=max(13, int(0.25 * fps) | 1))
    ang_vel_s[:calib_frames] = 0.0

    # ---------- Peak detection for complete stands ----------
    post = slice(calib_frames, None)
    post_h = stand_h_s[post]
    post_t = times[post]
    post_flex = trunk_flex_s[post]
    post_vel = ang_vel_s[post]

    h_range = float(np.nanmax(post_h) - np.nanmin(post_h)) if len(post_h) > 0 else 50.0
    thresh = max(18.0, 0.30 * h_range)          # adaptive height threshold (px)
    min_dist = max(int(0.65 * fps), 15)         # ≈ 0.65 s minimum between stands
    peaks_rel, props = find_peaks(post_h, height=thresh, distance=min_dist, prominence=12.0)
    n_stands = len(peaks_rel)
    peak_times = post_t[peaks_rel] if n_stands > 0 else np.array([])

    # Average time per stand (cycle time)
    if n_stands > 1:
        avg_time_per_stand = float(np.mean(np.diff(peak_times)))
    elif n_stands == 1:
        avg_time_per_stand = float(post_t[-1] - post_t[0])
    else:
        avg_time_per_stand = 0.0

    # ROM metrics
    peak_rom = float(np.nanmax(post_flex) - np.nanmin(post_flex)) if len(post_flex) > 0 else 0.0
    total_rom = float(np.nansum(np.abs(np.diff(post_flex)))) if len(post_flex) > 1 else 0.0   # total variation
    peak_ang_vel = float(np.nanmax(np.abs(post_vel))) if len(post_vel) > 0 else 0.0
    peak_vel_time = float(post_t[int(np.nanargmax(np.abs(post_vel)))]) if len(post_vel) > 0 else 0.0

    total_time = float(times[-1]) if len(times) else 0.0
    test_duration = min(30.0, total_time)   # standard 30CST window

    # -------------------- Results Summary --------------------
    print("=" * 70)
    print(" 30-SECOND CHAIR STAND TEST (Frontal Smartphone + MediaPipe Pose)")
    print(" Metrics: # complete stands | avg time/stand | hip/trunk ROM | ang. vel.")
    print("=" * 70)
    print(f" Total video time          : {total_time:6.2f} s")
    print(f" Calibration window        : {calib_frames/fps:6.2f} s")
    print(f" Analysis window           : {test_duration:6.2f} s")
    print()
    print(f" Number of complete stands : {n_stands:3d}")
    print(f" Average time per stand    : {avg_time_per_stand:6.2f} s")
    print(f" Peak hip/trunk ROM        : {peak_rom:6.1f} °")
    print(f" Total ROM (variation)     : {total_rom:6.1f} °")
    print(f" Peak angular velocity     : {peak_ang_vel:6.1f} °/s @ {peak_vel_time:5.2f} s")
    print()
    if n_stands > 0:
        print(f" Stand peak times (s)      : {np.round(peak_times, 2).tolist()}")
    print("=" * 70)
    print()

    # -------------------- CSV --------------------
    df = pd.DataFrame({
        "frame": np.arange(len(times)),
        "time_s": np.round(times, 3),
        "stand_height_px": np.round(stand_h_s, 1),          # positive = risen
        "trunk_flexion_deg": np.round(trunk_flex_s, 2),     # 0 ≈ upright
        "angular_velocity_deg_s": np.round(ang_vel_s, 1)
    })
    with open(OUTPUT_CSV, "w") as f:
        f.write("# 30-Second Chair Stand Test – Frontal Smartphone + MediaPipe Pose\n")
        f.write("# Initial ~2 s used for seated baseline calibration\n")
        f.write("# stand_height_px > 0 means hips risen above seated baseline\n")
        f.write("# trunk_flexion_deg approximates forward lean via projected trunk length\n")
        f.write("# Units: time = s, height = px, angle = degrees, velocity = deg/s\n")
        f.write(f"# Number of complete stands = {n_stands}\n")
        f.write(f"# Average time per stand = {avg_time_per_stand:.2f} s\n")
        f.write(f"# Peak hip/trunk ROM = {peak_rom:.1f} deg\n")
        f.write(f"# Total ROM (variation) = {total_rom:.1f} deg\n")
        f.write(f"# Peak angular velocity = {peak_ang_vel:.1f} deg/s\n\n")
    df.to_csv(OUTPUT_CSV, mode="a", index=False)

    summary_rows = [
        {"Metric": "Number of complete stands", "Value": n_stands, "Unit": "count"},
        {"Metric": "Average time per stand", "Value": round(avg_time_per_stand, 2), "Unit": "seconds"},
        {"Metric": "Peak hip/trunk ROM", "Value": round(peak_rom, 1), "Unit": "degrees"},
        {"Metric": "Total ROM (variation)", "Value": round(total_rom, 1), "Unit": "degrees"},
        {"Metric": "Peak angular velocity", "Value": round(peak_ang_vel, 1), "Unit": "degrees/second"},
        {"Metric": "Time of peak velocity", "Value": round(peak_vel_time, 2), "Unit": "seconds"},
        {"Metric": "Test duration analysed", "Value": round(test_duration, 2), "Unit": "seconds"},
    ]
    summary_df = pd.DataFrame(summary_rows)
    with open(OUTPUT_CSV, "a") as f:
        f.write("\n# Results Summary\n")
    summary_df.to_csv(OUTPUT_CSV, mode="a", index=False)
    print(f"CSV saved → {OUTPUT_CSV}")

    # -------------------- Annotated Video (continuous smooth overlay) --------------------
    # Pre-compute cumulative stand count at each frame for live display
    cum_stands = np.zeros(len(times), dtype=int)
    for p in peaks_rel:
        abs_idx = p + calib_frames
        if abs_idx < len(cum_stands):
            cum_stands[abs_idx:] += 1

    peak_frame_set = set(p + calib_frames for p in peaks_rel)

    cap = cv2.VideoCapture(str(video_path))
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    out = cv2.VideoWriter(str(OUTPUT_VIDEO), fourcc, fps, (w, h))

    with PoseLandmarker.create_from_options(options) as landmarker:
        idx = 0
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            mp_img = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
            res = landmarker.detect_for_video(mp_img, int(idx * 1000 / fps))

            # Draw keypoints (hips, shoulders, knees)
            if res.pose_landmarks and len(res.pose_landmarks) > 0:
                lms = res.pose_landmarks[0]
                for i, color in [
                    (11, (0, 255, 120)), (12, (0, 255, 120)),   # shoulders
                    (23, (0, 200, 255)), (24, (0, 200, 255)),   # hips
                    (25, (255, 180, 0)), (26, (255, 180, 0)),   # knees
                ]:
                    try:
                        if lms[i].visibility > 0.35:
                            x = int(lms[i].x * w)
                            y = int(lms[i].y * h)
                            cv2.circle(frame, (x, y), 5, color, -1)
                    except Exception:
                        pass
                # mid-hip to mid-shoulder line
                try:
                    if (lms[11].visibility > 0.4 and lms[12].visibility > 0.4 and
                        lms[23].visibility > 0.4 and lms[24].visibility > 0.4):
                        mids = (int(0.5*(lms[11].x+lms[12].x)*w), int(0.5*(lms[11].y+lms[12].y)*h))
                        midh = (int(0.5*(lms[23].x+lms[24].x)*w), int(0.5*(lms[23].y+lms[24].y)*h))
                        cv2.line(frame, mids, midh, (0, 220, 255), 2)
                except Exception:
                    pass

            # Smooth continuous overlay panel
            overlay = frame.copy()
            panel_h = 210
            cv2.rectangle(overlay, (6, 6), (480, panel_h), (20, 20, 20), -1)
            cv2.addWeighted(overlay, 0.68, frame, 0.32, 0, frame)
            cv2.rectangle(frame, (6, 6), (480, panel_h), (0, 220, 255), 2)

            t_cur = times[idx] if idx < len(times) else 0.0
            h_cur = stand_h_s[idx] if idx < len(stand_h_s) else 0.0
            ang_cur = trunk_flex_s[idx] if idx < len(trunk_flex_s) else 0.0
            vel_cur = abs(ang_vel_s[idx]) if idx < len(ang_vel_s) else 0.0
            n_cur = int(cum_stands[idx]) if idx < len(cum_stands) else n_stands

            font = cv2.FONT_HERSHEY_SIMPLEX
            y0, dy = 30, 26
            cv2.putText(frame, "30s Chair Stand – Frontal", (14, y0),
                        font, 0.58, (0, 255, 255), 2)
            cv2.putText(frame, f"Time: {t_cur:5.2f} s", (14, y0 + dy),
                        font, 0.58, (255, 255, 255), 2)
            cv2.putText(frame, f"Stands: {n_cur:2d}  (final {n_stands})", (14, y0 + 2*dy),
                        font, 0.58, (80, 255, 120), 2)
            cv2.putText(frame, f"Trunk flex: {ang_cur:5.1f} deg", (14, y0 + 3*dy),
                        font, 0.55, (80, 220, 255), 2)
            cv2.putText(frame, f"Ang. vel: {vel_cur:5.1f} deg/s", (14, y0 + 4*dy),
                        font, 0.55, (0, 200, 255), 2)
            cv2.putText(frame, f"Peak ROM: {peak_rom:4.1f} deg  |  Peak vel: {peak_ang_vel:5.1f}",
                        (14, y0 + 5*dy + 4), font, 0.48, (180, 255, 180), 1)
            cv2.putText(frame, f"Avg time/stand: {avg_time_per_stand:.2f} s",
                        (14, y0 + 6*dy + 2), font, 0.48, (180, 255, 180), 1)

            if idx in peak_frame_set:
                cv2.putText(frame, "STAND", (w - 110, 40), font, 0.7, (0, 0, 255), 2)
                cv2.circle(frame, (w - 30, 70), 12, (0, 0, 255), -1)

            if idx < calib_frames:
                cv2.putText(frame, "CALIBRATION (seated)", (w - 250, h - 20),
                            font, 0.50, (0, 255, 100), 2)

            out.write(frame)
            idx += 1

    cap.release()
    out.release()
    print(f"Annotated video saved → {OUTPUT_VIDEO}\n")
    return summary_df

# ===================== RUN =====================
summary = process_30s_chair_stand(VIDEO_PATH)
print(summary)

Video: 464×832 @ 59.9 fps (1857 frames)



W0000 00:00:1787502144.868114     334 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1787502144.892121     336 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


Calibration window: first 119 frames (~1.99 s)
Baseline hip_y (seated): 448.7 px
Baseline trunk projected length: 153.2 px
 30-SECOND CHAIR STAND TEST (Frontal Smartphone + MediaPipe Pose)
 Metrics: # complete stands | avg time/stand | hip/trunk ROM | ang. vel.
 Total video time          :  30.97 s
 Calibration window        :   1.99 s
 Analysis window           :  30.00 s

 Number of complete stands :  12
 Average time per stand    :   2.35 s
 Peak hip/trunk ROM        :   43.8 °
 Total ROM (variation)     : 1061.0 °
 Peak angular velocity     :  208.4 °/s @  2.62 s

 Stand peak times (s)      : [3.14, 5.61, 8.03, 9.98, 12.63, 15.05, 17.67, 20.08, 22.1, 24.57, 27.02, 29.02]

CSV saved → /kaggle/working/30s_ChairStand_metrics.csv


W0000 00:00:1787502179.289318     347 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1787502179.317636     347 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


Annotated video saved → /kaggle/working/30s_ChairStand_annotated.mp4

                      Metric    Value            Unit
0  Number of complete stands    12.00           count
1     Average time per stand     2.35         seconds
2         Peak hip/trunk ROM    43.80         degrees
3      Total ROM (variation)  1061.00         degrees
4      Peak angular velocity   208.40  degrees/second
5      Time of peak velocity     2.62         seconds
6     Test duration analysed    30.00         seconds
